In [1]:
# 1. pip install
%pip install pandas numpy requests jupyter python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 🚧 서울시 도로굴착 공사현황 데이터 전처리
# FR-005: 음성 안내 서비스 - 데이터 전처리 단계

import pandas as pd
import numpy as np
import requests
import json
import os
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()


True

In [3]:

print("🚧 FR-005: 도로굴착 공사현황 데이터 전처리")
print("🎯 단순화 버전 - 핵심 정보만 활용")
print("🔐 환경변수 로드 완료")
print("=" * 70)

# =============================================================================
# 0. 카카오 API 연결 테스트
# =============================================================================

def test_kakao_api():
    """카카오 API 연결 상태 확인"""
    print("🔍 카카오 API 연결 테스트...")
    
    api_key = os.getenv("KAKAO_REST_API_KEY")
    print(f"🔑 API 키: {api_key[:10] if api_key else 'None'}...")
    
    if not api_key:
        print("❌ API 키가 설정되지 않았습니다.")
        return False
    
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"query": "서울시 강남구"}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data.get('documents'):
                first_result = data['documents'][0]
                lat = first_result.get('y')
                lng = first_result.get('x')
                print(f"✅ API 연결 성공!")
                print(f"   📍 테스트 결과: 위도 {lat}, 경도 {lng}")
                return True
            else:
                print("❌ API 응답 데이터 없음")
                return False
        else:
            print(f"❌ API 오류: {response.status_code}")
            print(f"   응답: {response.text[:100]}")
            return False
            
    except Exception as e:
        print(f"❌ API 테스트 실패: {str(e)}")
        return False

# API 테스트 실행
api_available = test_kakao_api()
print(f"🌐 API 사용 가능: {'Yes' if api_available else 'No (구별 중심좌표 사용)'}")
print("-" * 70)


🚧 FR-005: 도로굴착 공사현황 데이터 전처리
🎯 단순화 버전 - 핵심 정보만 활용
🔐 환경변수 로드 완료
🔍 카카오 API 연결 테스트...
🔑 API 키: 7730cd19fd...
✅ API 연결 성공!
   📍 테스트 결과: 위도 37.5173319258532, 경도 127.047377408384
🌐 API 사용 가능: Yes
----------------------------------------------------------------------


In [4]:

# =============================================================================
# 1. 데이터 로드 및 기본 정보 확인
# =============================================================================

def load_road_excavation_data(file_path="../../data/csv/서울시 도로굴착 공사 현황.csv"):
    """
    도로굴착 현황 CSV 파일 로드
    
    Args:
        file_path (str): CSV 파일 경로
    
    Returns:
        pd.DataFrame: 로드된 데이터프레임
    """
    try:
        # 여러 인코딩으로 시도
        encodings = ['cp949', 'euc-kr', 'utf-8']
        
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"✅ 파일 로드 성공 (인코딩: {encoding})")
                break
            except UnicodeDecodeError:
                continue
        else:
            raise Exception("모든 인코딩 시도 실패")
            
        # 기본 정보 출력
        print(f"📊 총 데이터: {len(df):,}건")
        print(f"📋 원본 컬럼: {len(df.columns)}개")
        print(f"📅 컬럼 목록: {list(df.columns)}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        print("💡 파일 경로를 확인하거나 다음 경로들을 시도해보세요:")
        possible_paths = [
            "data/csv/서울시 도로굴착현황.csv",
            "./서울시 도로굴착현황.csv",
            "../data/csv/서울시 도로굴착현황.csv"
        ]
        for path in possible_paths:
            print(f"   - {path}")
        return None
    except Exception as e:
        print(f"❌ 파일 로드 중 오류 발생: {str(e)}")
        return None

# 데이터 로드
print("\n📂 데이터 로드 시작...")
df_raw = load_road_excavation_data()

if df_raw is not None:
    print(f"\n📋 데이터 미리보기 (첫 3행):")
    display_cols = df_raw.columns[:5]  # 첫 5개 컬럼만 표시
    print(df_raw[display_cols].head(3))
    
    # 자치구별 분포 확인
    if '구' in df_raw.columns:
        print(f"\n🏢 자치구별 분포 (상위 10개):")
        district_counts = df_raw['구'].value_counts()
        for district, count in district_counts.head(10).items():
            print(f"   - {district}: {count:,}건")
        print(f"   총 {len(district_counts)}개 자치구, 전체 {district_counts.sum():,}건")



📂 데이터 로드 시작...
✅ 파일 로드 성공 (인코딩: cp949)
📊 총 데이터: 533건
📋 원본 컬럼: 10개
📅 컬럼 목록: ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', '신청자', '처리상태', '도로', '포장', '허가번호']

📋 데이터 미리보기 (첫 3행):
           관리코드    구    동                                                공사명  \
0  일반-210728-75  서초구  잠원동  잠원동 신반포14차조합(롯데건설)3000kW 신설_보도(대선, 송영아)<BR>(서초...   
1  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   
2  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   

                 착공일 ~ 준공일  
0  2022-06-20 ~ 2022-08-31  
1  2022-06-16 ~ 2022-08-14  
2  2022-06-16 ~ 2022-08-14  

🏢 자치구별 분포 (상위 10개):
   - 서초구: 192건
   - 강남구: 94건
   - 성북구: 37건
   - 관악구: 33건
   - 강동구: 33건
   - 강북구: 19건
   - 종로구: 19건
   - 도봉구: 18건
   - 금천구: 18건
   - 노원구: 15건
   총 23개 자치구, 전체 533건


In [5]:
# =============================================================================
# 2. 단순화된 데이터 전처리
# =============================================================================

def create_simplified_construction_data(df_raw):
    """
    핵심 정보만 추출하여 단순화된 공사 데이터 생성
    
    Args:
        df_raw (pd.DataFrame): 원본 데이터프레임
    
    Returns:
        pd.DataFrame: 단순화된 데이터프레임
    """
    print("\n🎯 단순화된 공사 데이터 생성...")
    
    # 1. 핵심 컬럼만 선택
    essential_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일']
    
    # 컬럼 존재 여부 확인
    missing_cols = [col for col in essential_cols if col not in df_raw.columns]
    if missing_cols:
        print(f"❌ 필수 컬럼 누락: {missing_cols}")
        print(f"📋 사용 가능한 컬럼: {list(df_raw.columns)}")
        return None
    
    df_simple = df_raw[essential_cols].copy()
    print(f"✅ 핵심 컬럼 추출 완료: {len(df_simple)}건")
    
    # 2. HTML 태그 제거 및 주소 추출
    print("🧹 텍스트 정제 중...")
    df_simple['공사명_정제'] = df_simple['공사명'].str.replace('<BR>', ' ', regex=False)
    df_simple['공사명_정제'] = df_simple['공사명_정제'].str.replace('<br>', ' ', regex=False)
    
    # 괄호 안 주소 추출 (다양한 패턴 시도)
    patterns = [
        r'\(([^)]*구[^)]*동[^)]*)\)',  # 구+동 패턴
        r'\(([^)]*구[^)]*)\)',        # 구만 있는 패턴
        r'\(([^)]*\d+[^)]*)\)'        # 번지수 패턴
    ]
    
    df_simple['추출주소'] = None
    for i, pattern in enumerate(patterns):
        mask = df_simple['추출주소'].isna()
        if mask.sum() > 0:
            extracted = df_simple.loc[mask, '공사명_정제'].str.extract(pattern)[0]
            df_simple.loc[mask, '추출주소'] = extracted
            print(f"   패턴 {i+1}: {extracted.notna().sum()}건 추출")
    
    extracted_count = df_simple['추출주소'].notna().sum()
    print(f"✅ 주소 추출 완료: {extracted_count:,}건 ({extracted_count/len(df_simple)*100:.1f}%)")
    
    # 3. 날짜 파싱
    print("📅 날짜 정보 파싱 중...")
    date_parts = df_simple['착공일 ~ 준공일'].str.split(' ~ ', expand=True)
    df_simple['착공일'] = pd.to_datetime(date_parts[0], errors='coerce')
    df_simple['준공일'] = pd.to_datetime(date_parts[1], errors='coerce')
    
    valid_start = df_simple['착공일'].notna().sum()
    valid_end = df_simple['준공일'].notna().sum()
    print(f"✅ 날짜 파싱 완료: 착공일 {valid_start:,}건, 준공일 {valid_end:,}건")
    
    # 날짜 범위 확인
    if valid_start > 0:
        min_date = df_simple['착공일'].min()
        max_date = df_simple['준공일'].max()
        print(f"   📅 날짜 범위: {min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')}")
    
    # 4. 현재 공사 상태 계산 (단순하게 3가지만)
    print("🔍 공사 상태 계산 중...")
    today = datetime.now()
    
    conditions = [
        df_simple['착공일'] > today,  # 미착공
        (df_simple['착공일'] <= today) & (df_simple['준공일'] >= today),  # 진행중
        df_simple['준공일'] < today   # 완료
    ]
    choices = ['미착공', '진행중', '완료']
    df_simple['공사상태'] = np.select(conditions, choices, default='불명')
    
    print("✅ 공사 상태 분류 완료:")
    status_counts = df_simple['공사상태'].value_counts()
    for status, count in status_counts.items():
        print(f"   - {status}: {count:,}건 ({count/len(df_simple)*100:.1f}%)")
    
    # 5. 지오코딩용 주소 생성
    print("🗺️ 지오코딩용 주소 생성 중...")
    
    # 우선순위: 추출주소 > 구+동 조합
    mask_extracted = df_simple['추출주소'].notna()
    df_simple.loc[mask_extracted, '지오코딩주소'] = '서울시 ' + df_simple.loc[mask_extracted, '추출주소']
    df_simple.loc[~mask_extracted, '지오코딩주소'] = '서울시 ' + df_simple.loc[~mask_extracted, '구'] + ' ' + df_simple.loc[~mask_extracted, '동']
    
    print(f"✅ 지오코딩 주소 생성 완료:")
    print(f"   - 추출주소 기반: {mask_extracted.sum():,}건")
    print(f"   - 구+동 기반: {(~mask_extracted).sum():,}건")
    
    # 지오코딩 주소 샘플 출력
    print(f"   📍 주소 샘플:")
    for i, addr in enumerate(df_simple['지오코딩주소'].head(3), 1):
        print(f"      {i}. {addr}")
    
    # 6. 최종 컬럼 정리
    final_cols = ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태']
    df_final = df_simple[final_cols].copy()
    
    print(f"\n📊 단순화 완료:")
    print(f"   - 최종 데이터: {len(df_final):,}건")
    print(f"   - 최종 컬럼: {list(df_final.columns)}")
    
    return df_final

# 단순화된 전처리 실행
if df_raw is not None:
    df_simplified = create_simplified_construction_data(df_raw)
    
    if df_simplified is not None:
        print(f"\n📋 단순화된 데이터 미리보기:")
        print(df_simplified.head(3))


🎯 단순화된 공사 데이터 생성...
✅ 핵심 컬럼 추출 완료: 533건
🧹 텍스트 정제 중...
   패턴 1: 533건 추출
✅ 주소 추출 완료: 533건 (100.0%)
📅 날짜 정보 파싱 중...
✅ 날짜 파싱 완료: 착공일 372건, 준공일 372건
   📅 날짜 범위: 2020-08-27 ~ 2026-07-14
🔍 공사 상태 계산 중...
✅ 공사 상태 분류 완료:
   - 완료: 338건 (63.4%)
   - 불명: 161건 (30.2%)
   - 진행중: 34건 (6.4%)
🗺️ 지오코딩용 주소 생성 중...
✅ 지오코딩 주소 생성 완료:
   - 추출주소 기반: 533건
   - 구+동 기반: 0건
   📍 주소 샘플:
      1. 서울시 서초구 잠원동   잠원동112 ~ 잠원동   잠원동112
      2. 서울시 용산구 동자동   35-41 ~ 동자동   35-41부근
      3. 서울시 용산구 동자동   19-87 ~ 동자동   19-87부근

📊 단순화 완료:
   - 최종 데이터: 533건
   - 최종 컬럼: ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태']

📋 단순화된 데이터 미리보기:
           관리코드                               지오코딩주소        착공일        준공일  \
0  일반-210728-75  서울시 서초구 잠원동   잠원동112 ~ 잠원동   잠원동112 2022-06-20 2022-08-31   
1  일반-210527-84  서울시 용산구 동자동   35-41 ~ 동자동   35-41부근 2022-06-16 2022-08-14   
2  일반-210527-84  서울시 용산구 동자동   19-87 ~ 동자동   19-87부근 2022-06-16 2022-08-14   

  공사상태  
0   완료  
1   완료  
2   완료  


In [6]:
# =============================================================================
# 3. 지오코딩 (카카오맵 API + 백업 좌표)
# =============================================================================

class SimpleKakaoGeocoder:
    """단순화된 카카오맵 지오코딩 서비스"""
    
    def __init__(self):
        self.api_key = os.getenv('KAKAO_REST_API_KEY')
        self.base_url = "https://dapi.kakao.com/v2/local/search/address.json"
        self.delay = 0.2  # API 호출 간격 (안전하게 설정)
        
        # 서울시 자치구별 중심 좌표 (API 실패 시 사용)
        self.district_coords = {
            '강남구': (37.4979, 127.0276), '강동구': (37.5301, 127.1238),
            '강북구': (37.6398, 127.0256), '강서구': (37.5509, 126.8495),
            '관악구': (37.4784, 126.9516), '광진구': (37.5385, 127.0823),
            '구로구': (37.4955, 126.8874), '금천구': (37.4563, 126.8956),
            '노원구': (37.6542, 127.0568), '도봉구': (37.6688, 127.0471),
            '동대문구': (37.5744, 127.0396), '동작구': (37.5124, 126.9393),
            '마포구': (37.5637, 126.9084), '서대문구': (37.5794, 126.9368),
            '서초구': (37.4837, 127.0324), '성동구': (37.5634, 127.0365),
            '성북구': (37.5894, 127.0167), '송파구': (37.5145, 127.1059),
            '양천구': (37.5168, 126.8664), '영등포구': (37.5264, 126.8962),
            '용산구': (37.5326, 126.9905), '은평구': (37.6028, 126.9292),
            '종로구': (37.5735, 126.9788), '중구': (37.5641, 126.9979),
            '중랑구': (37.6063, 127.0925)
        }
    
    def get_coordinates(self, address):
        """단일 주소를 좌표로 변환"""
        if not self.api_key:
            return None
            
        try:
            headers = {"Authorization": f"KakaoAK {self.api_key}"}
            params = {"query": address}
            
            time.sleep(self.delay)
            response = requests.get(self.base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    result = data['documents'][0]
                    lat, lng = float(result['y']), float(result['x'])
                    
                    # 서울시 범위 검증 (대략적)
                    if 37.4 <= lat <= 37.7 and 126.8 <= lng <= 127.2:
                        return {'lat': lat, 'lng': lng, 'status': 'API성공'}
                    else:
                        return {'lat': lat, 'lng': lng, 'status': 'API성공(범위외)'}
                else:
                    return {'lat': None, 'lng': None, 'status': 'API결과없음'}
            elif response.status_code == 429:
                print("   ⚠️ API 호출 한도 초과 - 잠시 대기...")
                time.sleep(5)
                return self.get_coordinates(address)  # 재시도
            else:
                return {'lat': None, 'lng': None, 'status': f'API오류({response.status_code})'}
                
        except Exception as e:
            return {'lat': None, 'lng': None, 'status': f'예외({str(e)[:20]})'}
    
    def get_district_center(self, address):
        """구별 중심 좌표 반환 (API 실패 시 사용)"""
        for district, coords in self.district_coords.items():
            if district in address:
                lat, lng = coords
                # 구 내에서 약간의 랜덤 변화 (±1km 정도)
                lat += np.random.uniform(-0.01, 0.01)
                lng += np.random.uniform(-0.01, 0.01)
                return {'lat': lat, 'lng': lng, 'status': f'구중심({district})'}
        
        # 서울시 중심 (알 수 없는 경우)
        return {'lat': 37.5665, 'lng': 126.9780, 'status': '서울중심'}

def convert_addresses_to_coordinates(df, batch_size=30, sample_size=None):
    """
    지오코딩 실행 (API + 백업 좌표)
    
    Args:
        df (pd.DataFrame): 주소가 포함된 데이터프레임
        batch_size (int): 배치 크기 (API 안정성을 위해 줄임)
        sample_size (int): 테스트용 샘플 크기 (None이면 전체 처리)
    
    Returns:
        pd.DataFrame: 좌표가 추가된 데이터프레임
    """
    df_coord = df.copy()
    geocoder = SimpleKakaoGeocoder()
    
    # 샘플링 (테스트용)
    if sample_size and sample_size < len(df_coord):
        print(f"🧪 테스트용 샘플링: {sample_size}건만 처리")
        df_coord = df_coord.head(sample_size)
    
    print(f"\n🗺️ 지오코딩 시작... (총 {len(df_coord):,}건)")
    
    if not geocoder.api_key:
        print("⚠️  API 키 없음 → 구별 중심 좌표 사용")
        return add_district_coordinates(df_coord, geocoder)
    
    print(f"✅ API 사용 가능 → 실제 지오코딩 진행")
    print(f"📍 배치 크기: {batch_size}개씩 처리")
    
    # 배치별 API 호출
    all_results = []
    failed_count = 0
    
    total_batches = (len(df_coord) + batch_size - 1) // batch_size
    
    for i in range(0, len(df_coord), batch_size):
        batch_end = min(i + batch_size, len(df_coord))
        batch = df_coord.iloc[i:batch_end]
        batch_num = i // batch_size + 1
        
        print(f"   배치 {batch_num}/{total_batches}: {i+1}-{batch_end} ({len(batch)}개)")
        
        for idx, row in batch.iterrows():
            # API 시도
            result = geocoder.get_coordinates(row['지오코딩주소'])
            
            if result and result['lat'] is not None:
                all_results.append({
                    'index': idx,
                    'lat': result['lat'],
                    'lng': result['lng'],
                    'status': result['status']
                })
            else:
                # API 실패 시 구별 중심 좌표 사용
                failed_count += 1
                backup = geocoder.get_district_center(row['지오코딩주소'])
                all_results.append({
                    'index': idx,
                    'lat': backup['lat'],
                    'lng': backup['lng'],
                    'status': backup['status']
                })
        
        # 배치 간 대기 (API 안정성)
        if batch_num < total_batches:
            time.sleep(1)
    
    # 결과 병합
    coord_df = pd.DataFrame(all_results)
    df_result = df_coord.merge(coord_df, left_index=True, right_on='index', how='left')
    df_result.drop('index', axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    # 결과 요약
    print("✅ 지오코딩 완료:")
    status_counts = df_result['좌표상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_result) * 100) if len(df_result) > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    # 좌표 범위 확인
    valid_coords = df_result['위도'].notna().sum()
    if valid_coords > 0:
        lat_range = f"{df_result['위도'].min():.4f} ~ {df_result['위도'].max():.4f}"
        lng_range = f"{df_result['경도'].min():.4f} ~ {df_result['경도'].max():.4f}"
        print(f"   🗺️ 좌표 범위: 위도 {lat_range}, 경도 {lng_range}")
    
    return df_result

def add_district_coordinates(df, geocoder):
    """구별 중심 좌표만 사용하는 백업 방법"""
    
    coordinates = []
    for idx, row in df.iterrows():
        coord = geocoder.get_district_center(row['지오코딩주소'])
        coordinates.append({
            'index': idx,
            'lat': coord['lat'],
            'lng': coord['lng'],
            'status': coord['status']
        })
    
    coord_df = pd.DataFrame(coordinates)
    df_result = df.merge(coord_df, left_index=True, right_on='index', how='left')
    df_result.drop('index', axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    print(f"✅ 구별 중심 좌표 할당 완료: {len(df_result)}건")
    return df_result

# 지오코딩 실행
if 'df_simplified' in locals() and df_simplified is not None:
    # 테스트를 위해 처음 100건만 처리 (전체 처리하려면 sample_size=None)
    print("\n💡 테스트를 위해 처음 100건만 처리합니다.")
    print("   전체 처리하려면 아래 코드에서 sample_size=None으로 변경하세요.")
    
    df_with_coords = convert_addresses_to_coordinates(df_simplified, batch_size=50, sample_size=None)
    
    if df_with_coords is not None:
        print(f"\n📍 좌표 변환 결과 미리보기:")
        display_cols = ['관리코드', '지오코딩주소', '공사상태', '위도', '경도', '좌표상태']
        print(df_with_coords[display_cols].head(5))


💡 테스트를 위해 처음 100건만 처리합니다.
   전체 처리하려면 아래 코드에서 sample_size=None으로 변경하세요.

🗺️ 지오코딩 시작... (총 533건)
✅ API 사용 가능 → 실제 지오코딩 진행
📍 배치 크기: 50개씩 처리
   배치 1/11: 1-50 (50개)


KeyboardInterrupt: 

In [ ]:
df_with_coords.head(10)

,관리코드,지오코딩주소,착공일,준공일,공사상태,위도,경도,좌표상태
0,일반-210728-75,서울시 서초구 잠원동 잠원동112 ~ 잠원동 잠원동112,2022-06-20,2022-08-31,완료,37.507930,127.003492,API성공
1,일반-210527-84,서울시 용산구 동자동 35-41 ~ 동자동 35-41부근,2022-06-16,2022-08-14,완료,37.523145,126.982056,구중심(용산구)
2,일반-210527-84,서울시 용산구 동자동 19-87 ~ 동자동 19-87부근,2022-06-16,2022-08-14,완료,37.542405,126.994573,구중심(용산구)
3,일반-210907-41,서울시 관악구 봉천동 958-14 ~ 봉천동 958-14,2022-06-15,2022-12-30,완료,37.484925,126.936693,API성공
4,인터넷-220604-01,서울시 종로구 이화동 149 ~ 이화동 149,2022-06-15,2022-06-19,완료,37.577306,127.003633,API성공
5,일반-210525-89,서울시 송파구 잠실동 183-4 ~ 잠실동 183-4,2022-06-13,2022-07-31,완료,37.511030,127.084156,API성공
6,일반-220527-35,서울시 종로구 홍지동 127-36 ~ 홍지동 127-36,2022-06-13,2022-07-30,완료,37.597624,126.959110,API성공
7,일반-210416-80,서울시 강남구 대치동 942-3 ~ 대치동 942-3,2022-06-02,2022-06-30,완료,37.505865,127.057580,API성공
8,일반-210416-20,서울시 관악구 봉천동 912-26 ~ 봉천동 912-26,2022-05-26,2022-12-30,완료,37.482305,126.945666,API성공
9,일반-210506-07,서울시 강남구 청담동 77-39 ~ 청담동 77-39,2022-05-26,2022-06-30,완료,37.522872,127.048431,API성공


In [ ]:
# =============================================================================
# 4. 최종 검증 및 저장
# =============================================================================

def validate_and_save_final_data(df, output_dir="../../data/processed"):
    """
    최종 데이터 검증 및 저장
    
    Args:
        df (pd.DataFrame): 최종 처리된 데이터프레임
        output_dir (str): 저장할 디렉터리
    
    Returns:
        dict: 검증 결과 요약
    """
    print(f"\n🔍 최종 데이터 검증...")
    
    # 1. 기본 통계
    total_records = len(df)
    valid_coords = (df['위도'].notna() & df['경도'].notna()).sum()
    valid_dates = (df['착공일'].notna() & df['준공일'].notna()).sum()
    ongoing_projects = (df['공사상태'] == '진행중').sum()
    
    summary = {
        'total_records': total_records,
        'valid_coordinates': valid_coords,
        'valid_dates': valid_dates,
        'ongoing_projects': ongoing_projects,
        'coord_success_rate': (valid_coords / total_records * 100) if total_records > 0 else 0,
        'date_success_rate': (valid_dates / total_records * 100) if total_records > 0 else 0
    }
    
    # 2. 좌표 범위 검증 (서울시 범위)
    if valid_coords > 0:
        seoul_coords = (
            (df['위도'] >= 37.4) & (df['위도'] <= 37.7) &
            (df['경도'] >= 126.8) & (df['경도'] <= 127.2)
        ).sum()
        summary['seoul_range_coords'] = seoul_coords
        summary['seoul_range_rate'] = (seoul_coords / valid_coords * 100) if valid_coords > 0 else 0
    
    # 3. 결과 출력
    print("=" * 60)
    print("📊 최종 데이터 요약")
    print("=" * 60)
    print(f"🎯 총 데이터: {summary['total_records']:,}건")
    print(f"📍 유효 좌표: {summary['valid_coordinates']:,}건 ({summary['coord_success_rate']:.1f}%)")
    print(f"📅 유효 날짜: {summary['valid_dates']:,}건 ({summary['date_success_rate']:.1f}%)")
    print(f"🚧 진행중 공사: {summary['ongoing_projects']:,}건")
    
    if 'seoul_range_coords' in summary:
        print(f"🗺️ 서울시 범위: {summary['seoul_range_coords']:,}건 ({summary['seoul_range_rate']:.1f}%)")
    
    print(f"\n🏢 공사 상태별 분포:")
    status_counts = df['공사상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / total_records * 100) if total_records > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    if '좌표상태' in df.columns:
        print(f"\n📍 좌표 획득 방법별 분포:")
        coord_status_counts = df['좌표상태'].value_counts()
        for status, count in coord_status_counts.items():
            percentage = (count / total_records * 100) if total_records > 0 else 0
            print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    # 4. 데이터 저장
    try:
        os.makedirs(output_dir, exist_ok=True)
        
        # CSV 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"road_excavation_latest.csv"
        csv_path = os.path.join(output_dir, csv_filename)
        
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        file_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
        
        print(f"\n💾 데이터 저장 완료:")
        print(f"   - 파일: {csv_path}")
        print(f"   - 크기: {file_size_mb:.2f} MB")
        print(f"   - 컬럼: {list(df.columns)}")
        
        summary['output_file'] = csv_path
        summary['file_size_mb'] = file_size_mb
        
        # # 최신 파일 링크 생성
        # latest_path = os.path.join(output_dir, "road_excavation_latest.csv")
        # df.to_csv(latest_path, index=False, encoding='utf-8-sig')
        # print(f"   - 최신버전: {latest_path}")
        
    except Exception as e:
        print(f"❌ 저장 실패: {str(e)}")
        summary['save_error'] = str(e)
    
    return summary

# 최종 검증 및 저장
if 'df_with_coords' in locals() and df_with_coords is not None:
    final_summary = validate_and_save_final_data(df_with_coords)
    
    print(f"\n🎉 FR-005 데이터 전처리 완료!")
    print("=" * 70)
    print("📋 최종 데이터셋 정보:")
    print(f"   - 총 레코드: {final_summary['total_records']:,}건")
    print(f"   - 좌표 성공률: {final_summary['coord_success_rate']:.1f}%")
    print(f"   - 날짜 성공률: {final_summary['date_success_rate']:.1f}%")
    print(f"   - 현재 진행중 공사: {final_summary['ongoing_projects']:,}건")
    
    if 'output_file' in final_summary:
        print(f"   - 저장 파일: {final_summary['output_file']}")
        print(f"   - 파일 크기: {final_summary['file_size_mb']:.2f} MB")
    
    print("\n🚀 다음 단계:")
    print("   1. 음성 안내 메시지 생성 로직 개발")
    print("   2. 사용자 위치 기반 공사 현장 필터링")
    print("   3. Azure Speech Services 연동")
    print("   4. 실시간 경로 안내 시스템 구축")

else:
    print("❌ 전처리 과정에서 오류가 발생했습니다.")
    print("💡 위의 단계들을 다시 확인해보세요.")
    print("\n🔍 문제 해결 방법:")
    print("   1. CSV 파일 경로 확인")
    print("   2. 필수 컬럼 존재 여부 확인")
    print("   3. 카카오 API 키 설정 확인")
    print("   4. 인터넷 연결 상태 확인")


🔍 최종 데이터 검증...
📊 최종 데이터 요약
🎯 총 데이터: 533건
📍 유효 좌표: 533건 (100.0%)
📅 유효 날짜: 372건 (69.8%)
🚧 진행중 공사: 34건
🗺️ 서울시 범위: 533건 (100.0%)

🏢 공사 상태별 분포:
   - 완료: 338건 (63.4%)
   - 불명: 161건 (30.2%)
   - 진행중: 34건 (6.4%)

📍 좌표 획득 방법별 분포:
   - API성공: 433건 (81.2%)
   - 구중심(강남구): 45건 (8.4%)
   - 구중심(강북구): 14건 (2.6%)
   - 구중심(성북구): 12건 (2.3%)
   - 구중심(서초구): 5건 (0.9%)
   - 구중심(동대문구): 5건 (0.9%)
   - 구중심(용산구): 4건 (0.8%)
   - 구중심(종로구): 3건 (0.6%)
   - 구중심(서대문구): 3건 (0.6%)
   - 구중심(관악구): 2건 (0.4%)
   - 구중심(구로구): 2건 (0.4%)
   - 구중심(강동구): 1건 (0.2%)
   - 구중심(노원구): 1건 (0.2%)
   - 구중심(마포구): 1건 (0.2%)
   - 구중심(은평구): 1건 (0.2%)
   - 구중심(송파구): 1건 (0.2%)

💾 데이터 저장 완료:
   - 파일: ../../data/processed\road_excavation_processed_20250623_143840.csv
   - 크기: 0.08 MB
   - 컬럼: ['관리코드', '지오코딩주소', '착공일', '준공일', '공사상태', '위도', '경도', '좌표상태']
   - 최신버전: ../../data/processed\road_excavation_latest.csv

🎉 FR-005 데이터 전처리 완료!
📋 최종 데이터셋 정보:
   - 총 레코드: 533건
   - 좌표 성공률: 100.0%
   - 날짜 성공률: 69.8%
   - 현재 진행중 공사: 34건
   - 저장 파일: ../../data/pr

In [ ]:
# =============================================================================
# 5. 추가 분석 및 시각화 (선택사항)
# =============================================================================

def analyze_construction_patterns(df):
    """공사 패턴 분석 (선택적 실행)"""
    
    if df is None or len(df) == 0:
        print("❌ 분석할 데이터가 없습니다.")
        return
    
    print(f"\n📊 공사 패턴 분석...")
    
    # 1. 월별 공사 시작 패턴
    if '착공일' in df.columns and df['착공일'].notna().sum() > 0:
        df['착공월'] = df['착공일'].dt.to_period('M')
        monthly_starts = df.groupby('착공월').size()
        
        print(f"📅 월별 착공 건수 (최근 12개월):")
        for month, count in monthly_starts.tail(12).items():
            print(f"   - {month}: {count:,}건")
    
    # 2. 공사 기간 분석
    if '착공일' in df.columns and '준공일' in df.columns:
        df['공사기간_일'] = (df['준공일'] - df['착공일']).dt.days
        valid_duration = df['공사기간_일'].dropna()
        
        if len(valid_duration) > 0:
            print(f"\n⏱️ 공사 기간 통계:")
            print(f"   - 평균: {valid_duration.mean():.0f}일")
            print(f"   - 중간값: {valid_duration.median():.0f}일")
            print(f"   - 최단: {valid_duration.min():.0f}일")
            print(f"   - 최장: {valid_duration.max():.0f}일")
    
    # 3. 지역별 공사 밀도
    if '좌표상태' in df.columns:
        coord_success = df.groupby('좌표상태').size()
        print(f"\n🗺️ 좌표 획득 현황:")
        for status, count in coord_success.items():
            print(f"   - {status}: {count:,}건")
    
    # 4. 현재 진행중인 공사 정보
    ongoing = df[df['공사상태'] == '진행중']
    if len(ongoing) > 0:
        print(f"\n🚧 현재 진행중인 공사 ({len(ongoing):,}건):")
        
        # 완료 예정일 기준 정렬
        if '준공일' in ongoing.columns:
            upcoming = ongoing.dropna(subset=['준공일']).sort_values('준공일')
            print(f"   📅 곧 완료 예정 (상위 5개):")
            for i, (_, row) in enumerate(upcoming.head(5).iterrows(), 1):
                days_left = (row['준공일'] - datetime.now()).days
                print(f"      {i}. {row.get('지오코딩주소', 'N/A')[:30]}... ({days_left}일 남음)")
    
    return df

# 패턴 분석 실행 (선택사항)
if 'df_with_coords' in locals() and df_with_coords is not None:
    print("\n" + "=" * 70)
    print("📈 추가 분석 실행 (선택사항)")
    print("=" * 70)
    
    try:
        df_analyzed = analyze_construction_patterns(df_with_coords)
    except Exception as e:
        print(f"⚠️ 분석 중 오류 발생: {str(e)}")

print("\n" + "=" * 70)
print("🏁 FR-005.ipynb 전체 실행 완료")
print("=" * 70)
print("💡 성공적으로 처리되었다면 다음 파일들이 생성됩니다:")
print("   - data/processed/road_excavation_processed_YYYYMMDD_HHMMSS.csv")
print("   - data/processed/road_excavation_latest.csv")


📈 추가 분석 실행 (선택사항)

📊 공사 패턴 분석...
📅 월별 착공 건수 (최근 12개월):
   - 2021-01: 1건
   - 2021-07: 7건
   - 2021-08: 16건
   - 2021-09: 8건
   - 2021-11: 1건
   - 2021-12: 16건
   - 2022-01: 17건
   - 2022-02: 2건
   - 2022-03: 20건
   - 2022-04: 119건
   - 2022-05: 131건
   - 2022-06: 8건

⏱️ 공사 기간 통계:
   - 평균: 406일
   - 중간값: 190일
   - 최단: 4일
   - 최장: 1845일

🗺️ 좌표 획득 현황:
   - API성공: 433건
   - 구중심(강남구): 45건
   - 구중심(강동구): 1건
   - 구중심(강북구): 14건
   - 구중심(관악구): 2건
   - 구중심(구로구): 2건
   - 구중심(노원구): 1건
   - 구중심(동대문구): 5건
   - 구중심(마포구): 1건
   - 구중심(서대문구): 3건
   - 구중심(서초구): 5건
   - 구중심(성북구): 12건
   - 구중심(송파구): 1건
   - 구중심(용산구): 4건
   - 구중심(은평구): 1건
   - 구중심(종로구): 3건

🚧 현재 진행중인 공사 (34건):
   📅 곧 완료 예정 (상위 5개):
      1. 서울시 강남구 청담동   77도 ~ 청담동   77-4... (6일 남음)
      2. 서울시 노원구 중계동   한글비석로262 ~ 상계동  ... (105일 남음)
      3. 서울시 성북구 하월곡동   230-2 ~ 하월곡동   ... (190일 남음)
      4. 서울시 성북구 하월곡동   오패산로 1가길 17-1, ... (190일 남음)
      5. 서울시 성북구 종암동   종암로 127-1, 3-173... (190일 남음)

🏁 FR-005.ipynb 전체 실행 완료
💡 성공적으로 처리되었다면 다음 파일들이 생성

In [7]:
# 🚧 서울시 도로굴착 공사현황 데이터 전처리 (개선 버전)
# FR-005: 음성 안내 서비스 - 위험도 분류 및 분석 포함

import pandas as pd
import numpy as np
import requests
import json
import os
import time
import re
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import threading

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()

print("🚧 FR-005: 도로굴착 공사현황 데이터 전처리 (개선 버전)")
print("🎯 위험도 분류 및 분석 포함")
print("🔐 환경변수 로드 완료")
print("=" * 70)

# =============================================================================
# 0. 카카오 API 연결 테스트
# =============================================================================

def test_kakao_api():
    """카카오 API 연결 상태 확인"""
    print("🔍 카카오 API 연결 테스트...")
    
    api_key = os.getenv("KAKAO_REST_API_KEY")
    print(f"🔑 API 키: {api_key[:10] if api_key else 'None'}...")
    
    if not api_key:
        print("❌ API 키가 설정되지 않았습니다.")
        return False
    
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"query": "서울시 강남구"}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data.get('documents'):
                first_result = data['documents'][0]
                lat = first_result.get('y')
                lng = first_result.get('x')
                print(f"✅ API 연결 성공!")
                print(f"   📍 테스트 결과: 위도 {lat}, 경도 {lng}")
                return True
            else:
                print("❌ API 응답 데이터 없음")
                return False
        else:
            print(f"❌ API 오류: {response.status_code}")
            print(f"   응답: {response.text[:100]}")
            return False
            
    except Exception as e:
        print(f"❌ API 테스트 실패: {str(e)}")
        return False

# API 테스트 실행
api_available = test_kakao_api()
print(f"🌐 API 사용 가능: {'Yes' if api_available else 'No (구별 중심좌표 사용)'}")
print("-" * 70)

# =============================================================================
# 1. 데이터 로드 및 기본 정보 확인
# =============================================================================

def load_road_excavation_data(file_path="../../data/csv/서울시 도로굴착 공사 현황.csv"):
    """
    도로굴착 현황 CSV 파일 로드 (UTF-8 버전)
    
    Args:
        file_path (str): CSV 파일 경로
    
    Returns:
        pd.DataFrame: 로드된 데이터프레임
    """
    try:
        # UTF-8로 읽기
        df = pd.read_csv(file_path, encoding='utf-8')
        print(f"✅ 파일 로드 성공 (UTF-8)")
        
        # 기본 정보 출력
        print(f"📊 총 데이터: {len(df):,}건")
        print(f"📋 컬럼: {len(df.columns)}개")
        print(f"📅 컬럼 목록: {list(df.columns)}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return None
    except Exception as e:
        print(f"❌ 파일 로드 중 오류 발생: {str(e)}")
        return None

# 데이터 로드
print("\n📂 데이터 로드 시작...")
df_raw = load_road_excavation_data()

if df_raw is not None:
    print(f"\n📋 데이터 미리보기 (첫 3행):")
    display_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일']
    print(df_raw[display_cols].head(3))
    
    # 자치구별 분포 확인
    if '구' in df_raw.columns:
        print(f"\n🏢 자치구별 분포 (상위 10개):")
        district_counts = df_raw['구'].value_counts()
        for district, count in district_counts.head(10).items():
            print(f"   - {district}: {count:,}건")
        print(f"   총 {len(district_counts)}개 자치구, 전체 {district_counts.sum():,}건")

# =============================================================================
# 2. 공사 유형 자동 분류 시스템
# =============================================================================

class ConstructionClassifier:
    """공사명 기반 자동 분류 시스템"""
    
    def __init__(self):
        # 공사 유형별 키워드 규칙
        self.type_rules = {
            '전력/전기': {
                'keywords': ['전주', '전력', '전기', 'kw', 'kva', '한국전력', '전선', '변압기', 
                           '지중화', '인입', '신설', '철거', '이설'],
                'risk_level': 3,
                'risk_category': '보통',
                'description': '전주교체, 지중화, 전력설비'
            },
            '통신': {
                'keywords': ['통신', '케이블', '광케이블', '전화', 'kt', 'lg', 'sk', 
                           '인터넷', '5g', 'lte'],
                'risk_level': 2,
                'risk_category': '낮음',
                'description': '통신선로, 얕은 굴착'
            },
            '상하수도': {
                'keywords': ['상하수도', '하수관', '상수관', '급수', '배수', '하수', 
                           '상수', '수도관', '배관'],
                'risk_level': 4,
                'risk_category': '높음',
                'description': '깊은 굴착, 지하매설물'
            },
            '가스': {
                'keywords': ['가스', 'lpg', 'lng', '도시가스', '가스관'],
                'risk_level': 5,
                'risk_category': '매우높음',
                'description': '폭발위험, 특별주의'
            },
            '도로': {
                'keywords': ['도로', '포장', '아스팔트', '콘크리트', '보수', '개량', 
                           '재포장', '노면'],
                'risk_level': 2,
                'risk_category': '낮음',
                'description': '표면작업, 교통불편'
            }
        }
        
        # 도로 등급 매핑
        self.road_grade_mapping = {
            '특별시도': '대로',
            '시도': '중로',
            '구도': '소로',
            '기타': '기타'
        }
        
        # 영향반경 추정 (미터)
        self.impact_radius = {
            '전력/전기': 50,
            '통신': 30,
            '상하수도': 100,
            '가스': 200,
            '도로': 20,
            '기타': 50
        }
    
    def classify_construction_type(self, construction_name):
        """공사명을 기반으로 공사 유형 분류"""
        if pd.isna(construction_name):
            return '기타'
        
        name_lower = str(construction_name).lower()
        
        for type_name, info in self.type_rules.items():
            if any(keyword in name_lower for keyword in info['keywords']):
                return type_name
        
        return '기타'
    
    def get_construction_info(self, construction_type):
        """공사 유형별 상세 정보 반환"""
        if construction_type in self.type_rules:
            return self.type_rules[construction_type]
        else:
            return {
                'risk_level': 3,
                'risk_category': '보통',
                'description': '분류되지 않은 공사'
            }
    
    def get_road_grade(self, road_type):
        """도로 유형을 등급으로 변환"""
        return self.road_grade_mapping.get(road_type, '기타')
    
    def get_impact_radius(self, construction_type):
        """공사 유형별 영향반경 반환"""
        return self.impact_radius.get(construction_type, 50)

# =============================================================================
# 3. 개선된 데이터 전처리
# =============================================================================

def create_enhanced_construction_data(df_raw):
    """
    개선된 공사 데이터 생성 (위험도 분류 포함)
    
    Args:
        df_raw (pd.DataFrame): 원본 데이터프레임
    
    Returns:
        pd.DataFrame: 개선된 데이터프레임
    """
    print("\n🎯 개선된 공사 데이터 생성...")
    
    # 분류기 초기화
    classifier = ConstructionClassifier()
    
    # 1. 기본 컬럼 선택 (허가번호 제외)
    essential_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', 
                     '처리상태', '도로', '포장']
    
    # 컬럼 존재 여부 확인
    missing_cols = [col for col in essential_cols if col not in df_raw.columns]
    if missing_cols:
        print(f"❌ 필수 컬럼 누락: {missing_cols}")
        return None
    
    df_enhanced = df_raw[essential_cols].copy()
    print(f"✅ 기본 컬럼 추출 완료: {len(df_enhanced)}건")
    
    # 2. HTML 태그 제거 및 주소 추출
    print("🧹 텍스트 정제 중...")
    df_enhanced['공사명_정제'] = df_enhanced['공사명'].str.replace('<BR>', ' ', regex=False)
    df_enhanced['공사명_정제'] = df_enhanced['공사명_정제'].str.replace('<br>', ' ', regex=False)
    
    # 괄호 안 주소 추출
    df_enhanced['지오코딩주소'] = df_enhanced['공사명'].str.extract(r'\(([^)]*구[^)]*동[^)]*)\)')
    df_enhanced['지오코딩주소'] = '서울시 ' + df_enhanced['지오코딩주소'].fillna('')
    
    # 주소 추출 실패 시 구+동 조합 사용
    mask_failed = df_enhanced['지오코딩주소'].str.len() <= 4
    df_enhanced.loc[mask_failed, '지오코딩주소'] = '서울시 ' + df_enhanced.loc[mask_failed, '구'] + ' ' + df_enhanced.loc[mask_failed, '동']
    
    extracted_count = df_enhanced['지오코딩주소'].notna().sum()
    print(f"✅ 주소 추출 완료: {extracted_count:,}건")
    
    # 3. 날짜 파싱
    print("📅 날짜 정보 파싱 중...")
    date_parts = df_enhanced['착공일 ~ 준공일'].str.split(' ~ ', expand=True)
    df_enhanced['착공일'] = pd.to_datetime(date_parts[0], errors='coerce')
    df_enhanced['준공일'] = pd.to_datetime(date_parts[1], errors='coerce')
    
    valid_dates = df_enhanced['착공일'].notna().sum()
    print(f"✅ 날짜 파싱 완료: {valid_dates:,}건")
    
    # 4. 공사 상태 계산
    print("🔍 공사 상태 계산 중...")
    today = datetime.now()
    
    conditions = [
        df_enhanced['착공일'] > today,  # 미착공
        (df_enhanced['착공일'] <= today) & (df_enhanced['준공일'] >= today),  # 진행중
        df_enhanced['준공일'] < today   # 완료
    ]
    choices = ['미착공', '진행중', '완료']
    df_enhanced['공사상태'] = np.select(conditions, choices, default='불명')
    
    status_counts = df_enhanced['공사상태'].value_counts()
    print("✅ 공사 상태 분류 완료:")
    for status, count in status_counts.items():
        print(f"   - {status}: {count:,}건 ({count/len(df_enhanced)*100:.1f}%)")
    
    # 5. 공사 유형 자동 분류
    print("🤖 공사 유형 자동 분류 중...")
    df_enhanced['공사유형'] = df_enhanced['공사명'].apply(classifier.classify_construction_type)
    
    # 대분류 생성
    df_enhanced['공사유형_대분류'] = df_enhanced['공사유형'].apply(
        lambda x: x if x != '기타' else '기타'
    )
    
    type_counts = df_enhanced['공사유형'].value_counts()
    print("✅ 공사 유형 분류 완료:")
    for type_name, count in type_counts.items():
        print(f"   - {type_name}: {count:,}건 ({count/len(df_enhanced)*100:.1f}%)")
    
    # 6. 위험도 정보 추가
    print("🚨 위험도 정보 생성 중...")
    construction_info = df_enhanced['공사유형'].apply(classifier.get_construction_info)
    
    df_enhanced['위험도_레벨'] = construction_info.apply(lambda x: x['risk_level'])
    df_enhanced['위험도_카테고리'] = construction_info.apply(lambda x: x['risk_category'])
    
    risk_distribution = df_enhanced['위험도_카테고리'].value_counts()
    print("✅ 위험도 분류 완료:")
    for category, count in risk_distribution.items():
        print(f"   - {category}: {count:,}건")
    
    # 7. 도로 등급 및 영향반경
    print("🛣️ 도로 정보 및 영향반경 계산 중...")
    df_enhanced['도로등급'] = df_enhanced['도로'].apply(classifier.get_road_grade)
    df_enhanced['영향반경_미터'] = df_enhanced['공사유형'].apply(classifier.get_impact_radius)
    
    road_grade_counts = df_enhanced['도로등급'].value_counts()
    print("✅ 도로 등급 분류 완료:")
    for grade, count in road_grade_counts.items():
        print(f"   - {grade}: {count:,}건")
    
    # 8. 최종 컬럼 정리
    final_cols = [
        # 기본 정보 (7개)
        '관리코드', '구', '동', '공사명', '공사명_정제', '처리상태',
        
        # 날짜 및 상태 (3개)
        '착공일', '준공일', '공사상태',
        
        # 위치 정보 (1개)
        '지오코딩주소',
        
        # 도로/포장 정보 (2개)
        '도로', '포장',
        
        # 분류 및 위험도 (6개)
        '공사유형', '공사유형_대분류', '위험도_레벨', '위험도_카테고리', 
        '도로등급', '영향반경_미터'
    ]
    
    df_final = df_enhanced[final_cols].copy()
    
    print(f"\n📊 개선된 데이터 생성 완료:")
    print(f"   - 최종 데이터: {len(df_final):,}건")
    print(f"   - 최종 컬럼: {len(final_cols)}개")
    
    return df_final

# 개선된 전처리 실행
if df_raw is not None:
    df_enhanced = create_enhanced_construction_data(df_raw)
    
    if df_enhanced is not None:
        print(f"\n📋 개선된 데이터 미리보기:")
        display_cols = ['관리코드', '구', '공사유형', '위험도_레벨', '위험도_카테고리']
        print(df_enhanced[display_cols].head(5))

# =============================================================================
# 4. 지오코딩 (기존 코드와 동일)
# =============================================================================

class SimpleKakaoGeocoder:
    """카카오맵 지오코딩 서비스 (병렬처리 최적화)"""
    
    def __init__(self):
        self.api_key = os.getenv('KAKAO_REST_API_KEY')
        self.base_url = "https://dapi.kakao.com/v2/local/search/address.json"
        self.delay = 0.1  # 병렬처리로 인한 딜레이 감소
        self.session = requests.Session()  # 세션 재사용으로 성능 향상
        self.lock = Lock()  # 스레드 안전성
        self.request_count = 0
        self.success_count = 0
        
        # 서울시 자치구별 중심 좌표
        self.district_coords = {
            '강남구': (37.4979, 127.0276), '강동구': (37.5301, 127.1238),
            '강북구': (37.6398, 127.0256), '강서구': (37.5509, 126.8495),
            '관악구': (37.4784, 126.9516), '광진구': (37.5385, 127.0823),
            '구로구': (37.4955, 126.8874), '금천구': (37.4563, 126.8956),
            '노원구': (37.6542, 127.0568), '도봉구': (37.6688, 127.0471),
            '동대문구': (37.5744, 127.0396), '동작구': (37.5124, 126.9393),
            '마포구': (37.5637, 126.9084), '서대문구': (37.5794, 126.9368),
            '서초구': (37.4837, 127.0324), '성동구': (37.5634, 127.0365),
            '성북구': (37.5894, 127.0167), '송파구': (37.5145, 127.1059),
            '양천구': (37.5168, 126.8664), '영등포구': (37.5264, 126.8962),
            '용산구': (37.5326, 126.9905), '은평구': (37.6028, 126.9292),
            '종로구': (37.5735, 126.9788), '중구': (37.5641, 126.9979),
            '중랑구': (37.6063, 127.0925)
        }
    
    def get_coordinates_single(self, address, idx=None):
        """단일 주소를 좌표로 변환 (스레드 안전)"""
        with self.lock:
            self.request_count += 1
            
        if not self.api_key:
            return self._get_fallback_coordinates(address, idx)
            
        try:
            headers = {"Authorization": f"KakaoAK {self.api_key}"}
            params = {"query": address}
            
            # 병렬처리 시 너무 빠른 요청 방지
            time.sleep(self.delay)
            
            response = self.session.get(self.base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    result = data['documents'][0]
                    lat, lng = float(result['y']), float(result['x'])
                    
                    with self.lock:
                        self.success_count += 1
                    
                    if 37.4 <= lat <= 37.7 and 126.8 <= lng <= 127.2:
                        return {
                            'index': idx,
                            'lat': lat,
                            'lng': lng,
                            'status': 'API성공',
                            'address': address
                        }
                    else:
                        return {
                            'index': idx,
                            'lat': lat,
                            'lng': lng,
                            'status': 'API성공(범위외)',
                            'address': address
                        }
                else:
                    return self._get_fallback_coordinates(address, idx, 'API결과없음')
            elif response.status_code == 429:
                # 요청 한도 초과 시 잠시 대기 후 재시도
                time.sleep(2)
                return self.get_coordinates_single(address, idx)
            else:
                return self._get_fallback_coordinates(address, idx, f'API오류({response.status_code})')
                
        except Exception as e:
            return self._get_fallback_coordinates(address, idx, f'예외({str(e)[:20]})')
    
    def _get_fallback_coordinates(self, address, idx, status='API실패'):
        """백업 좌표 반환 (구별 중심점)"""
        for district, coords in self.district_coords.items():
            if district in address:
                lat, lng = coords
                # 약간의 랜덤 변화로 겹치지 않게
                lat += np.random.uniform(-0.01, 0.01)
                lng += np.random.uniform(-0.01, 0.01)
                return {
                    'index': idx,
                    'lat': lat,
                    'lng': lng,
                    'status': f'구중심({district})',
                    'address': address
                }
        
        # 구를 찾을 수 없으면 서울 중심
        return {
            'index': idx,
            'lat': 37.5665,
            'lng': 126.9780,
            'status': f'서울중심({status})',
            'address': address
        }
    
    def get_statistics(self):
        """API 호출 통계 반환"""
        return {
            'total_requests': self.request_count,
            'successful_requests': self.success_count,
            'success_rate': (self.success_count / self.request_count * 100) if self.request_count > 0 else 0
        }

def convert_addresses_to_coordinates_parallel(df, max_workers=10, batch_size=100, sample_size=None):
    """병렬 지오코딩 실행"""
    df_coord = df.copy()
    geocoder = SimpleKakaoGeocoder()
    
    if sample_size and sample_size < len(df_coord):
        print(f"🧪 테스트용 샘플링: {sample_size}건만 처리")
        df_coord = df_coord.head(sample_size)
    
    print(f"\n🗺️ 병렬 지오코딩 시작... (총 {len(df_coord):,}건)")
    print(f"⚡ 병렬 작업자: {max_workers}개 스레드")
    print(f"📦 배치 크기: {batch_size}건씩 처리")
    
    if not geocoder.api_key:
        print("⚠️ API 키 없음 → 구별 중심 좌표만 사용")
        max_workers = 1  # API 없을 때는 병렬처리 불필요
    
    # 주소와 인덱스 준비
    address_data = []
    for idx, row in df_coord.iterrows():
        address_data.append((row['지오코딩주소'], idx))
    
    # 배치별 병렬 처리
    all_results = []
    total_batches = (len(address_data) + batch_size - 1) // batch_size
    
    start_time = time.time()
    
    for batch_num in range(total_batches):
        batch_start = batch_num * batch_size
        batch_end = min(batch_start + batch_size, len(address_data))
        batch_data = address_data[batch_start:batch_end]
        
        print(f"   📦 배치 {batch_num + 1}/{total_batches}: {batch_start + 1}-{batch_end} ({len(batch_data)}개)")
        
        # 병렬 처리
        batch_results = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 작업 제출
            future_to_data = {
                executor.submit(geocoder.get_coordinates_single, address, idx): (address, idx)
                for address, idx in batch_data
            }
            
            # 결과 수집
            for future in as_completed(future_to_data):
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as e:
                    address, idx = future_to_data[future]
                    print(f"   ❌ 스레드 오류 {idx}: {str(e)[:50]}")
                    # 오류 시 백업 좌표 사용
                    backup = geocoder._get_fallback_coordinates(address, idx, '스레드오류')
                    batch_results.append(backup)
        
        all_results.extend(batch_results)
        
        # 배치 간 잠시 대기 (API 안정성)
        if batch_num < total_batches - 1 and geocoder.api_key:
            time.sleep(0.5)
        
        # 진행률 표시
        elapsed_time = time.time() - start_time
        processed = len(all_results)
        if processed > 0:
            estimated_total_time = elapsed_time * len(address_data) / processed
            remaining_time = estimated_total_time - elapsed_time
            print(f"   ⏱️ 진행률: {processed}/{len(address_data)} ({processed/len(address_data)*100:.1f}%) | "
                  f"경과: {elapsed_time:.1f}초 | 예상 남은시간: {remaining_time:.1f}초")
    
    # 결과 정리 및 병합
    print("\n🔄 결과 병합 중...")
    results_df = pd.DataFrame(all_results)
    
    # 인덱스 기준으로 원본 데이터와 병합
    df_result = df_coord.merge(results_df, left_index=True, right_on='index', how='left')
    df_result.drop(['index', 'address'], axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    # 통계 출력
    total_time = time.time() - start_time
    stats = geocoder.get_statistics()
    
    print("✅ 병렬 지오코딩 완료:")
    print(f"   ⏱️ 총 처리시간: {total_time:.1f}초")
    print(f"   🚀 평균 처리속도: {len(df_coord)/total_time:.1f}건/초")
    print(f"   📊 API 통계: {stats['total_requests']}회 요청, {stats['success_rate']:.1f}% 성공")
    
    # 좌표 상태별 분포
    status_counts = df_result['좌표상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_result) * 100) if len(df_result) > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    return df_result

# 지오코딩 실행 (병렬처리)
if 'df_enhanced' in locals() and df_enhanced is not None:
    print("\n💡 전체 데이터를 병렬처리로 처리합니다.")
    print("   ⚡ 병렬처리로 속도가 크게 향상됩니다!")
    
    # 병렬처리 설정
    max_workers = 8  # 동시 스레드 수 (너무 높으면 API 제한에 걸릴 수 있음)
    batch_size = 50  # 배치당 처리 건수
    
    df_with_coords = convert_addresses_to_coordinates_parallel(
        df_enhanced, 
        max_workers=max_workers, 
        batch_size=batch_size, 
        sample_size=None
    )
    
    if df_with_coords is not None:
        print(f"\n📍 좌표 변환 결과 미리보기:")
        display_cols = ['관리코드', '구', '공사유형', '위험도_레벨', '위도', '경도', '좌표상태']
        print(df_with_coords[display_cols].head(5))
    else:
        print("❌ 지오코딩 실패!")
else:
    print("❌ df_enhanced 데이터가 없습니다!")

# =============================================================================
# 5. 프로젝트별 집계 테이블 생성
# =============================================================================

def create_project_summary_table(df_with_coords):
    """프로젝트별 집계 테이블 생성 (중복 제거)"""
    print("\n📊 프로젝트별 집계 테이블 생성 중...")
    
    # 관리코드별 집계
    project_summary = []
    
    for code, group in df_with_coords.groupby('관리코드'):
        # 기본 정보 (첫 번째 행 기준)
        first_row = group.iloc[0]
        
        # 좌표 집계
        valid_coords = group.dropna(subset=['위도', '경도'])
        
        if len(valid_coords) > 0:
            center_lat = valid_coords['위도'].mean()
            center_lng = valid_coords['경도'].mean()
            
            # 최대 영향반경 계산 (중심점에서 가장 먼 구간까지의 거리)
            if len(valid_coords) > 1:
                distances = []
                for _, row in valid_coords.iterrows():
                    # 간단한 유클리드 거리 (실제로는 haversine 공식 사용 권장)
                    dist = np.sqrt((row['위도'] - center_lat)**2 + (row['경도'] - center_lng)**2) * 111000  # 대략적 미터 변환
                    distances.append(dist)
                max_impact_radius = max(distances) if distances else first_row['영향반경_미터']
            else:
                max_impact_radius = first_row['영향반경_미터']
        else:
            center_lat = center_lng = None
            max_impact_radius = first_row['영향반경_미터']
        
        # 가장 신뢰도 높은 좌표상태 선택
        coord_status_priority = {'API성공': 1, 'API성공(범위외)': 2, '구중심': 3, '서울중심': 4}
        best_status = min(group['좌표상태'].dropna(), 
                         key=lambda x: coord_status_priority.get(x.split('(')[0], 5))
        
        project_info = {
            # 기본 정보
            '관리코드': code,
            '구': first_row['구'],
            '동': first_row['동'],
            '공사명': first_row['공사명'],
            '공사명_정제': first_row['공사명_정제'],
            '처리상태': first_row['처리상태'],
            
            # 날짜 및 상태
            '착공일': first_row['착공일'],
            '준공일': first_row['준공일'],
            '공사상태': first_row['공사상태'],
            
            # 집계된 위치 정보
            '중심위도': center_lat,
            '중심경도': center_lng,
            '구간수': len(group),
            '최대영향반경': max_impact_radius,
            
            # 도로/포장 정보
            '도로': first_row['도로'],
            '포장': first_row['포장'],
            
            # 분류 및 위험도
            '공사유형': first_row['공사유형'],
            '공사유형_대분류': first_row['공사유형_대분류'],
            '위험도_레벨': first_row['위험도_레벨'],
            '위험도_카테고리': first_row['위험도_카테고리'],
            '도로등급': first_row['도로등급'],
            '영향반경_미터': first_row['영향반경_미터'],
            
            # 좌표 상태
            '최신_좌표상태': best_status
        }
        
        project_summary.append(project_info)
    
    df_projects = pd.DataFrame(project_summary)
    
    print(f"✅ 프로젝트 집계 완료:")
    print(f"   - 원본 행수: {len(df_with_coords):,}건")
    print(f"   - 집계 후: {len(df_projects):,}건 (중복 제거)")
    print(f"   - 중복률: {(1 - len(df_projects)/len(df_with_coords))*100:.1f}%")
    
    return df_projects

def create_segment_detail_table(df_with_coords):
    """구간 상세 테이블 생성"""
    print("\n📍 구간 상세 테이블 생성 중...")
    
    # 구간 ID 생성
    df_segments = df_with_coords.copy()
    df_segments['구간_ID'] = range(1, len(df_segments) + 1)
    
    # 구간별 순서 생성 (관리코드 내에서)
    df_segments['구간_순서'] = df_segments.groupby('관리코드').cumcount() + 1
    
    # 필요한 컬럼만 선택
    segment_cols = [
        '관리코드', '구간_ID', '구간_순서',
        '지오코딩주소', '위도', '경도', '좌표상태'
    ]
    
    df_segments_final = df_segments[segment_cols].copy()
    
    print(f"✅ 구간 상세 테이블 생성 완료:")
    print(f"   - 총 구간수: {len(df_segments_final):,}개")
    print(f"   - 컬럼수: {len(segment_cols)}개")
    
    return df_segments_final

# 집계 테이블 생성
if 'df_with_coords' in locals() and df_with_coords is not None:
    print("\n🔍 집계 테이블 생성 시작...")
    df_projects = create_project_summary_table(df_with_coords)
    df_segments = create_segment_detail_table(df_with_coords)
    
    if df_projects is not None and df_segments is not None:
        print(f"\n📋 프로젝트 요약 테이블 미리보기:")
        project_display_cols = ['관리코드', '구', '공사유형', '위험도_레벨', '구간수', '중심위도', '중심경도']
        print(df_projects[project_display_cols].head(3))
        
        print(f"\n📋 구간 상세 테이블 미리보기:")
        segment_display_cols = ['관리코드', '구간_ID', '구간_순서', '위도', '경도']
        print(df_segments[segment_display_cols].head(3))
    else:
        print("❌ 집계 테이블 생성 실패!")
else:
    print("❌ df_with_coords 데이터가 없습니다!")

# =============================================================================
# 6. 최종 검증 및 저장
# =============================================================================

def validate_and_save_enhanced_data(df_projects, df_segments, df_combined, output_dir="../../data/processed"):
    """개선된 데이터 검증 및 저장"""
    print(f"\n🔍 최종 데이터 검증...")
    
    # 1. 기본 통계
    total_projects = len(df_projects) if df_projects is not None else 0
    total_segments = len(df_segments) if df_segments is not None else 0
    total_combined = len(df_combined) if df_combined is not None else 0
    
    print("=" * 60)
    print("📊 최종 데이터 요약")
    print("=" * 60)
    print(f"🎯 프로젝트 요약: {total_projects:,}건")
    print(f"📍 구간 상세: {total_segments:,}건")
    print(f"🗂️ 통합 테이블: {total_combined:,}건")
    
    if df_projects is not None:
        # 현재 위험도 분포 (상태별)
        print(f"\n🚨 현재 위험도별 분포:")
        risk_dist = df_projects['현재_위험도_카테고리'].value_counts()
        for category, count in risk_dist.items():
            percentage = (count / total_projects * 100) if total_projects > 0 else 0
            print(f"   - {category}: {count:,}건 ({percentage:.1f}%)")
        
        # 공사 유형 분포
        print(f"\n🏗️ 공사 유형별 분포:")
        type_dist = df_projects['공사유형'].value_counts()
        for type_name, count in type_dist.head(5).items():
            percentage = (count / total_projects * 100) if total_projects > 0 else 0
            print(f"   - {type_name}: {count:,}건 ({percentage:.1f}%)")
        
        # 음성 안내 활성화 상태
        active_guidance = (df_projects['안내_활성화_여부'] == True).sum() if '안내_활성화_여부' in df_projects.columns else 0
        print(f"\n📢 음성 안내 활성화: {active_guidance:,}건 / {total_projects:,}건 ({active_guidance/total_projects*100:.1f}%)")
        
        # 상태별 진행중인 공사
        ongoing_projects = (df_projects['공사상태'] == '진행중').sum() if '공사상태' in df_projects.columns else 0
        planned_projects = (df_projects['공사상태'] == '미착공').sum() if '공사상태' in df_projects.columns else 0
        completed_projects = (df_projects['공사상태'] == '완료').sum() if '공사상태' in df_projects.columns else 0
        
        print(f"\n🚧 공사 상태별 분포:")
        print(f"   - 진행중: {ongoing_projects:,}건")
        print(f"   - 미착공: {planned_projects:,}건") 
        print(f"   - 완료: {completed_projects:,}건")
    
    

# 최종 검증 및 저장
print("\n🔍 최종 저장 단계 확인...")

# 변수 존재 여부를 더 확실하게 체크
def check_dataframe_exists(df_name, df_var):
    """데이터프레임 존재 여부와 상태 확인"""
    try:
        if df_var is not None and len(df_var) > 0:
            print(f"✅ {df_name}: {len(df_var)}건 존재")
            return True
        else:
            print(f"❌ {df_name}: 데이터가 없거나 빈 상태")
            return False
    except Exception as e:
        print(f"❌ {df_name}: 오류 - {str(e)}")
        return False

# 각 데이터프레임 상태 확인
try:
    projects_ok = check_dataframe_exists("df_projects", df_projects)
    segments_ok = check_dataframe_exists("df_segments", df_segments) 
    coords_ok = check_dataframe_exists("df_with_coords", df_with_coords)
    
    print(f"\n📊 데이터프레임 상태 요약:")
    print(f"   - 프로젝트 요약: {'✅' if projects_ok else '❌'}")
    print(f"   - 구간 상세: {'✅' if segments_ok else '❌'}")
    print(f"   - 통합 데이터: {'✅' if coords_ok else '❌'}")
    
    if projects_ok and segments_ok and coords_ok:
        print("\n🎉 모든 데이터프레임이 정상적으로 생성되었습니다!")
        print("💾 저장을 시작합니다...")
        
        try:
            final_summary = validate_and_save_enhanced_data(df_projects, df_segments, df_with_coords)
            
            if final_summary:
                print(f"\n🎉 FR-005 개선된 데이터 전처리 완료!")
                print("=" * 70)
                print("📋 최종 데이터셋 정보:")
                print(f"   - 프로젝트 요약: {final_summary['total_projects']:,}건")
                print(f"   - 구간 상세: {final_summary['total_segments']:,}건")
                print(f"   - 통합 데이터: {final_summary['total_combined']:,}건")
                print(f"   - 저장 파일: {len(final_summary['saved_files'])}개")
                
                print("\n📁 저장된 파일 목록:")
                for file_path in final_summary['saved_files']:
                    print(f"   - {file_path}")
                
                print("\n🚀 다음 단계:")
                print("   1. 음성 안내 메시지 템플릿 개발")
                print("   2. 위험도별 안전거리 설정")
                print("   3. 실시간 위치 기반 필터링 로직")
                print("   4. Azure Speech Services 연동")
                print("   5. 경로 안내 시스템 구축")
                
                print("\n📱 FR-005 음성 안내 활용 예시:")
                if len(df_projects) > 0:
                    high_risk_projects = df_projects[df_projects['위험도_레벨'] >= 4]
                    if len(high_risk_projects) > 0:
                        sample = high_risk_projects.iloc[0]
                        print(f"   🚨 고위험 공사 발견!")
                        print(f"   📍 위치: {sample['구']} {sample['동']}")
                        print(f"   🏗️ 유형: {sample['공사유형']} (위험도 {sample['위험도_레벨']})")
                        print(f"   📢 안내: '{sample['구']} {sample['동']} 지역에 {sample['공사유형']} 공사가 진행중입니다. 안전거리 {sample['영향반경_미터']}미터를 유지해주세요.'")
                    else:
                        print("   ℹ️ 현재 고위험(레벨 4-5) 공사는 없습니다.")
            else:
                print("❌ 저장 과정에서 오류가 발생했습니다.")
        except Exception as e:
            print(f"❌ 저장 중 예외 발생: {str(e)}")
            import traceback
            traceback.print_exc()
    else:
        print("\n❌ 일부 데이터프레임 생성에 실패했습니다.")
        print("💡 위의 단계들을 다시 확인해보세요.")

except NameError as e:
    print(f"❌ 변수 오류: {str(e)}")
    print("💡 이전 단계에서 일부 변수가 생성되지 않았습니다.")
except Exception as e:
    print(f"❌ 예상치 못한 오류: {str(e)}")
    import traceback
    traceback.print_exc()

# =============================================================================
# 7. 데이터 분석 및 인사이트
# =============================================================================

def analyze_construction_patterns(df_projects):
    """공사 패턴 심화 분석"""
    
    if df_projects is None or len(df_projects) == 0:
        print("❌ 분석할 프로젝트 데이터가 없습니다.")
        return
    
    print(f"\n📊 공사 패턴 심화 분석...")
    print("=" * 60)
    
    # 1. 위험도별 지역 분포
    print("🗺️ 위험도별 지역 분포:")
    risk_by_district = df_projects.groupby(['구', '위험도_카테고리']).size().unstack(fill_value=0)
    
    # 고위험 지역 TOP 5
    high_risk_districts = df_projects[df_projects['위험도_레벨'] >= 4]['구'].value_counts().head(5)
    print("   🚨 고위험 공사 다발 지역 (TOP 5):")
    for district, count in high_risk_districts.items():
        print(f"      - {district}: {count}건")
    
    # 2. 공사 유형별 평균 영향반경
    print(f"\n📏 공사 유형별 평균 영향반경:")
    avg_radius = df_projects.groupby('공사유형')['영향반경_미터'].mean().sort_values(ascending=False)
    for type_name, radius in avg_radius.items():
        print(f"   - {type_name}: {radius:.0f}미터")
    
    # 3. 도로 등급별 공사 분포
    print(f"\n🛣️ 도로 등급별 공사 분포:")
    road_dist = df_projects['도로등급'].value_counts()
    for grade, count in road_dist.items():
        percentage = (count / len(df_projects) * 100)
        print(f"   - {grade}: {count}건 ({percentage:.1f}%)")
    
    # 4. 진행중인 고위험 공사
    ongoing_high_risk = df_projects[
        (df_projects['공사상태'] == '진행중') & 
        (df_projects['위험도_레벨'] >= 4)
    ]
    
    print(f"\n🚧 현재 진행중인 고위험 공사: {len(ongoing_high_risk)}건")
    if len(ongoing_high_risk) > 0:
        print("   상위 3개 지역:")
        top_ongoing = ongoing_high_risk['구'].value_counts().head(3)
        for district, count in top_ongoing.items():
            print(f"   - {district}: {count}건")
    
    # 5. 음성 안내 우선순위 제안
    print(f"\n📢 음성 안내 우선순위 제안:")
    priority_score = df_projects.copy()
    priority_score['우선순위_점수'] = (
        priority_score['위험도_레벨'] * 2 +  # 위험도 가중치
        (priority_score['공사상태'] == '진행중').astype(int) * 3 +  # 진행중 가중치
        (priority_score['도로등급'] == '대로').astype(int) * 1  # 주요도로 가중치
    )
    
    top_priority = priority_score.nlargest(5, '우선순위_점수')
    for idx, row in top_priority.iterrows():
        print(f"   {idx+1}. [{row['관리코드']}] {row['구']} {row['동']}")
        print(f"      유형: {row['공사유형']} | 위험도: {row['위험도_레벨']} | 상태: {row['공사상태']}")
    
    return priority_score

# 패턴 분석 실행
if 'df_projects' in locals() and df_projects is not None:
    priority_data = analyze_construction_patterns(df_projects)

print("\n" + "=" * 70)
print("🏁 FR-005 개선된 전처리 스크립트 실행 완료")
print("=" * 70)
print("💡 생성된 파일들을 확인하고 음성 안내 로직 개발을 시작하세요!")
print("📧 문의사항이 있으시면 개발팀에 연락 부탁드립니다.")

🚧 FR-005: 도로굴착 공사현황 데이터 전처리 (개선 버전)
🎯 위험도 분류 및 분석 포함
🔐 환경변수 로드 완료
🔍 카카오 API 연결 테스트...
🔑 API 키: 7730cd19fd...
✅ API 연결 성공!
   📍 테스트 결과: 위도 37.5173319258532, 경도 127.047377408384
🌐 API 사용 가능: Yes
----------------------------------------------------------------------

📂 데이터 로드 시작...
✅ 파일 로드 성공 (UTF-8)
📊 총 데이터: 533건
📋 컬럼: 10개
📅 컬럼 목록: ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', '신청자', '처리상태', '도로', '포장', '허가번호']

📋 데이터 미리보기 (첫 3행):
           관리코드    구    동                                                공사명  \
0  일반-210728-75  서초구  잠원동  잠원동 신반포14차조합(롯데건설)3000kW 신설_보도(대선, 송영아)<BR>(서초...   
1  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   
2  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   

                 착공일 ~ 준공일  
0  2022-06-20 ~ 2022-08-31  
1  2022-06-16 ~ 2022-08-14  
2  2022-06-16 ~ 2022-08-14  

🏢 자치구별 분포 (상위 10개):
   - 서초구: 192건
   - 강남구: 94건
   - 성북구: 37건
   - 관악구: 33건
   - 강동구: 33건
   - 강북구: 19건
   - 종로구: 19건
   - 도봉구: 18건
   

Traceback (most recent call last):
  File "c:\Users\User\anaconda3\envs\py313\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: '현재_위험도_카테고리'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\User\AppData\Local\Temp\ipykernel_28400\1222941908.py", line 826, in <module>
    final_summary = validate_and_save_enhanced_data(df_projects, df_segments, df_with_coords)
  File "C:\Users\User\AppData\Local\Temp\ipykernel_28400\1222941908.p

In [8]:
# 🚧 서울시 도로굴착 공사현황 데이터 전처리 (개선 버전)
# FR-005: 음성 안내 서비스 - 위험도 분류 및 분석 포함

import pandas as pd
import numpy as np
import requests
import json
import os
import time
import re
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import threading

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()

print("🚧 FR-005: 도로굴착 공사현황 데이터 전처리 (개선 버전)")
print("🎯 위험도 분류 및 분석 포함")
print("🔐 환경변수 로드 완료")
print("=" * 70)

# =============================================================================
# 0. 카카오 API 연결 테스트
# =============================================================================

def test_kakao_api():
    """카카오 API 연결 상태 확인"""
    print("🔍 카카오 API 연결 테스트...")
    
    api_key = os.getenv("KAKAO_REST_API_KEY")
    print(f"🔑 API 키: {api_key[:10] if api_key else 'None'}...")
    
    if not api_key:
        print("❌ API 키가 설정되지 않았습니다.")
        return False
    
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"query": "서울시 강남구"}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data.get('documents'):
                first_result = data['documents'][0]
                lat = first_result.get('y')
                lng = first_result.get('x')
                print(f"✅ API 연결 성공!")
                print(f"   📍 테스트 결과: 위도 {lat}, 경도 {lng}")
                return True
            else:
                print("❌ API 응답 데이터 없음")
                return False
        else:
            print(f"❌ API 오류: {response.status_code}")
            print(f"   응답: {response.text[:100]}")
            return False
            
    except Exception as e:
        print(f"❌ API 테스트 실패: {str(e)}")
        return False

# API 테스트 실행
api_available = test_kakao_api()
print(f"🌐 API 사용 가능: {'Yes' if api_available else 'No (구별 중심좌표 사용)'}")
print("-" * 70)

# =============================================================================
# 1. 데이터 로드 및 기본 정보 확인
# =============================================================================

def load_road_excavation_data(file_path="../../data/csv/서울시 도로굴착 공사 현황.csv"):
    """
    도로굴착 현황 CSV 파일 로드 (UTF-8 버전)
    
    Args:
        file_path (str): CSV 파일 경로
    
    Returns:
        pd.DataFrame: 로드된 데이터프레임
    """
    try:
        # UTF-8로 읽기
        df = pd.read_csv(file_path, encoding='utf-8')
        print(f"✅ 파일 로드 성공 (UTF-8)")
        
        # 기본 정보 출력
        print(f"📊 총 데이터: {len(df):,}건")
        print(f"📋 컬럼: {len(df.columns)}개")
        print(f"📅 컬럼 목록: {list(df.columns)}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return None
    except Exception as e:
        print(f"❌ 파일 로드 중 오류 발생: {str(e)}")
        return None

# 데이터 로드
print("\n📂 데이터 로드 시작...")
df_raw = load_road_excavation_data()

if df_raw is not None:
    print(f"\n📋 데이터 미리보기 (첫 3행):")
    display_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일']
    print(df_raw[display_cols].head(3))
    
    # 자치구별 분포 확인
    if '구' in df_raw.columns:
        print(f"\n🏢 자치구별 분포 (상위 10개):")
        district_counts = df_raw['구'].value_counts()
        for district, count in district_counts.head(10).items():
            print(f"   - {district}: {count:,}건")
        print(f"   총 {len(district_counts)}개 자치구, 전체 {district_counts.sum():,}건")

# =============================================================================
# 2. 공사 유형 자동 분류 시스템
# =============================================================================

class ConstructionClassifier:
    """공사명 기반 자동 분류 시스템 (상태별 차등 위험도 포함)"""
    
    def __init__(self):
        # 공사 유형별 키워드 규칙 (기본 위험도)
        self.type_rules = {
            '전력/전기': {
                'keywords': ['전주', '전력', '전기', 'kw', 'kva', '한국전력', '전선', '변압기', 
                           '지중화', '인입', '신설', '철거', '이설'],
                'base_risk_level': 3,
                'risk_category_base': '보통',
                'description': '전주교체, 지중화, 전력설비'
            },
            '통신': {
                'keywords': ['통신', '케이블', '광케이블', '전화', 'kt', 'lg', 'sk', 
                           '인터넷', '5g', 'lte'],
                'base_risk_level': 2,
                'risk_category_base': '낮음',
                'description': '통신선로, 얕은 굴착'
            },
            '상하수도': {
                'keywords': ['상하수도', '하수관', '상수관', '급수', '배수', '하수', 
                           '상수', '수도관', '배관'],
                'base_risk_level': 4,
                'risk_category_base': '높음',
                'description': '깊은 굴착, 지하매설물'
            },
            '가스': {
                'keywords': ['가스', 'lpg', 'lng', '도시가스', '가스관'],
                'base_risk_level': 5,
                'risk_category_base': '매우높음',
                'description': '폭발위험, 특별주의'
            },
            '도로': {
                'keywords': ['도로', '포장', '아스팔트', '콘크리트', '보수', '개량', 
                           '재포장', '노면'],
                'base_risk_level': 2,
                'risk_category_base': '낮음',
                'description': '표면작업, 교통불편'
            }
        }
        
        # 상태별 위험도 가중치
        self.status_weight = {
            '미착공': 0.3,    # 예정 위험 (30%)
            '진행중': 1.0,    # 현재 위험 (100%)
            '완료': 0.2,      # 잔여 위험 (20%)
            '불명': 0.5       # 중간값 (50%)
        }
        
        # 안내 우선순위 (높을수록 우선)
        self.guidance_priority = {
            '진행중': 10,     # 최우선
            '미착공': 7,      # 사전 안내
            '불명': 5,        # 중간
            '완료': 3         # 낮은 우선순위
        }
        
        # 도로 등급 매핑
        self.road_grade_mapping = {
            '특별시도': '대로',
            '시도': '중로',
            '구도': '소로',
            '기타': '기타'
        }
        
        # 영향반경 추정 (미터)
        self.impact_radius = {
            '전력/전기': 50,
            '통신': 30,
            '상하수도': 100,
            '가스': 200,
            '도로': 20,
            '기타': 50
        }
    
    def classify_construction_type(self, construction_name):
        """공사명을 기반으로 공사 유형 분류"""
        if pd.isna(construction_name):
            return '기타'
        
        name_lower = str(construction_name).lower()
        
        for type_name, info in self.type_rules.items():
            if any(keyword in name_lower for keyword in info['keywords']):
                return type_name
        
        return '기타'
    
    def get_construction_info(self, construction_type):
        """공사 유형별 기본 정보 반환"""
        if construction_type in self.type_rules:
            return self.type_rules[construction_type]
        else:
            return {
                'base_risk_level': 3,
                'risk_category_base': '보통',
                'description': '분류되지 않은 공사'
            }
    
    def calculate_current_risk(self, base_risk_level, construction_status):
        """상태별 가중치를 적용한 현재 위험도 계산"""
        weight = self.status_weight.get(construction_status, 0.5)
        current_risk = base_risk_level * weight
        
        # 위험도 레벨을 1-5 범위로 제한 (소수점 포함)
        current_risk = max(0.1, min(5.0, current_risk))
        
        return round(current_risk, 1)
    
    def get_current_risk_category(self, current_risk_level, construction_status):
        """현재 위험도에 따른 카테고리 반환"""
        if construction_status == '완료':
            return '잔여위험'
        elif construction_status == '미착공':
            if current_risk_level >= 1.5:
                return '예정위험'
            else:
                return '낮은예정위험'
        else:  # 진행중, 불명
            if current_risk_level >= 4.0:
                return '매우높음'
            elif current_risk_level >= 3.0:
                return '높음'
            elif current_risk_level >= 2.0:
                return '보통'
            else:
                return '낮음'
    
    def get_guidance_priority(self, construction_status, current_risk_level):
        """음성 안내 우선순위 계산"""
        base_priority = self.guidance_priority.get(construction_status, 5)
        
        # 위험도에 따른 가산점
        risk_bonus = int(current_risk_level)
        
        return base_priority + risk_bonus
    
    def is_guidance_active(self, construction_status, current_risk_level):
        """음성 안내 대상 여부 판단"""
        # 완료된 공사 중 위험도가 매우 낮은 경우 비활성화
        if construction_status == '완료' and current_risk_level < 0.5:
            return False
        
        # 미착공 중 위험도가 매우 낮은 경우 비활성화
        if construction_status == '미착공' and current_risk_level < 0.8:
            return False
        
        # 나머지는 모두 활성화
        return True
    
    def get_road_grade(self, road_type):
        """도로 유형을 등급으로 변환"""
        return self.road_grade_mapping.get(road_type, '기타')
    
    def get_impact_radius(self, construction_type):
        """공사 유형별 영향반경 반환"""
        return self.impact_radius.get(construction_type, 50)
    
    def generate_guidance_message(self, district, dong, construction_type, construction_status, current_risk_level):
        """상태별 차별화된 안내 메시지 생성"""
        location = f"{district} {dong}"
        
        if construction_status == '진행중':
            if current_risk_level >= 4.0:
                return f"⚠️ {location} 지역에 {construction_type} 공사가 진행중입니다. 즉시 우회하시기 바랍니다."
            else:
                return f"🚧 {location} 지역에 {construction_type} 공사가 진행중입니다. 주의하여 통행하세요."
        
        elif construction_status == '미착공':
            return f"📅 {location} 지역에 {construction_type} 공사가 예정되어 있습니다. 향후 일정을 확인하세요."
        
        elif construction_status == '완료':
            if current_risk_level >= 0.5:
                return f"✅ {location} 지역 {construction_type} 공사가 완료되었습니다. 복구 상태를 확인하세요."
            else:
                return f"✅ {location} 지역 {construction_type} 공사가 완료되었습니다."
        
        else:  # 불명
            return f"❓ {location} 지역에 {construction_type} 공사 관련 정보가 있습니다. 현장 상황을 확인하세요."

# =============================================================================
# 3. 개선된 데이터 전처리
# =============================================================================

def create_enhanced_construction_data(df_raw):
    """
    개선된 공사 데이터 생성 (위험도 분류 포함)
    
    Args:
        df_raw (pd.DataFrame): 원본 데이터프레임
    
    Returns:
        pd.DataFrame: 개선된 데이터프레임
    """
    print("\n🎯 개선된 공사 데이터 생성...")
    
    # 분류기 초기화
    classifier = ConstructionClassifier()
    
    # 1. 기본 컬럼 선택 (허가번호 제외)
    essential_cols = ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', 
                     '처리상태', '도로', '포장']
    
    # 컬럼 존재 여부 확인
    missing_cols = [col for col in essential_cols if col not in df_raw.columns]
    if missing_cols:
        print(f"❌ 필수 컬럼 누락: {missing_cols}")
        return None
    
    df_enhanced = df_raw[essential_cols].copy()
    print(f"✅ 기본 컬럼 추출 완료: {len(df_enhanced)}건")
    
    # 2. HTML 태그 제거 및 주소 추출
    print("🧹 텍스트 정제 중...")
    df_enhanced['공사명_정제'] = df_enhanced['공사명'].str.replace('<BR>', ' ', regex=False)
    df_enhanced['공사명_정제'] = df_enhanced['공사명_정제'].str.replace('<br>', ' ', regex=False)
    
    # 괄호 안 주소 추출
    df_enhanced['지오코딩주소'] = df_enhanced['공사명'].str.extract(r'\(([^)]*구[^)]*동[^)]*)\)')
    df_enhanced['지오코딩주소'] = '서울시 ' + df_enhanced['지오코딩주소'].fillna('')
    
    # 주소 추출 실패 시 구+동 조합 사용
    mask_failed = df_enhanced['지오코딩주소'].str.len() <= 4
    df_enhanced.loc[mask_failed, '지오코딩주소'] = '서울시 ' + df_enhanced.loc[mask_failed, '구'] + ' ' + df_enhanced.loc[mask_failed, '동']
    
    extracted_count = df_enhanced['지오코딩주소'].notna().sum()
    print(f"✅ 주소 추출 완료: {extracted_count:,}건")
    
    # 3. 날짜 파싱
    print("📅 날짜 정보 파싱 중...")
    date_parts = df_enhanced['착공일 ~ 준공일'].str.split(' ~ ', expand=True)
    df_enhanced['착공일'] = pd.to_datetime(date_parts[0], errors='coerce')
    df_enhanced['준공일'] = pd.to_datetime(date_parts[1], errors='coerce')
    
    valid_dates = df_enhanced['착공일'].notna().sum()
    print(f"✅ 날짜 파싱 완료: {valid_dates:,}건")
    
    # 4. 공사 상태 계산
    print("🔍 공사 상태 계산 중...")
    today = datetime.now()
    
    conditions = [
        df_enhanced['착공일'] > today,  # 미착공
        (df_enhanced['착공일'] <= today) & (df_enhanced['준공일'] >= today),  # 진행중
        df_enhanced['준공일'] < today   # 완료
    ]
    choices = ['미착공', '진행중', '완료']
    df_enhanced['공사상태'] = np.select(conditions, choices, default='불명')
    
    status_counts = df_enhanced['공사상태'].value_counts()
    print("✅ 공사 상태 분류 완료:")
    for status, count in status_counts.items():
        print(f"   - {status}: {count:,}건 ({count/len(df_enhanced)*100:.1f}%)")
    
    # 5. 공사 유형 자동 분류
    print("🤖 공사 유형 자동 분류 중...")
    df_enhanced['공사유형'] = df_enhanced['공사명'].apply(classifier.classify_construction_type)
    
    # 대분류 생성
    df_enhanced['공사유형_대분류'] = df_enhanced['공사유형'].apply(
        lambda x: x if x != '기타' else '기타'
    )
    
    type_counts = df_enhanced['공사유형'].value_counts()
    print("✅ 공사 유형 분류 완료:")
    for type_name, count in type_counts.items():
        print(f"   - {type_name}: {count:,}건 ({count/len(df_enhanced)*100:.1f}%)")
    
    # 6. 개선된 위험도 정보 추가 (상태별 차등 적용)
    print("🚨 개선된 위험도 정보 생성 중...")
    
    # 기본 위험도 정보 가져오기
    construction_info = df_enhanced['공사유형'].apply(classifier.get_construction_info)
    
    # 기본 위험도 레벨 (공사 유형별 고정값)
    df_enhanced['기본_위험도_레벨'] = construction_info.apply(lambda x: x['base_risk_level'])
    df_enhanced['기본_위험도_카테고리'] = construction_info.apply(lambda x: x['risk_category_base'])
    
    # 현재 위험도 레벨 (상태별 가중치 적용)
    df_enhanced['현재_위험도_레벨'] = df_enhanced.apply(
        lambda row: classifier.calculate_current_risk(row['기본_위험도_레벨'], row['공사상태']), 
        axis=1
    )
    
    # 현재 위험도 카테고리
    df_enhanced['현재_위험도_카테고리'] = df_enhanced.apply(
        lambda row: classifier.get_current_risk_category(row['현재_위험도_레벨'], row['공사상태']), 
        axis=1
    )
    
    # 음성 안내 관련 정보
    df_enhanced['안내_우선순위'] = df_enhanced.apply(
        lambda row: classifier.get_guidance_priority(row['공사상태'], row['현재_위험도_레벨']), 
        axis=1
    )
    
    df_enhanced['안내_활성화_여부'] = df_enhanced.apply(
        lambda row: classifier.is_guidance_active(row['공사상태'], row['현재_위험도_레벨']), 
        axis=1
    )
    
    # 안내 메시지 생성
    df_enhanced['안내_메시지'] = df_enhanced.apply(
        lambda row: classifier.generate_guidance_message(
            row['구'], row['동'], row['공사유형'], row['공사상태'], row['현재_위험도_레벨']
        ), 
        axis=1
    )
    
    # 결과 요약
    print("✅ 개선된 위험도 분류 완료:")
    
    # 상태별 분포
    status_risk_summary = df_enhanced.groupby(['공사상태', '현재_위험도_카테고리']).size().unstack(fill_value=0)
    print("\n   📊 상태별 위험도 분포:")
    for status in ['진행중', '미착공', '완료', '불명']:
        if status in status_risk_summary.index:
            row_data = status_risk_summary.loc[status]
            total = row_data.sum()
            print(f"   - {status}: {total}건")
            for category, count in row_data[row_data > 0].items():
                print(f"     • {category}: {count}건")
    
    # 활성화된 안내 대상
    active_guidance = df_enhanced['안내_활성화_여부'].sum()
    total_count = len(df_enhanced)
    print(f"\n   📢 음성 안내 활성화: {active_guidance}건 / {total_count}건 ({active_guidance/total_count*100:.1f}%)")
    
    # 우선순위별 분포
    high_priority = (df_enhanced['안내_우선순위'] >= 10).sum()
    medium_priority = ((df_enhanced['안내_우선순위'] >= 7) & (df_enhanced['안내_우선순위'] < 10)).sum()
    low_priority = (df_enhanced['안내_우선순위'] < 7).sum()
    
    print(f"   🔥 높은 우선순위 (≥10): {high_priority}건")
    print(f"   🟡 중간 우선순위 (7-9): {medium_priority}건")
    print(f"   🔵 낮은 우선순위 (<7): {low_priority}건")
    
    # 7. 도로 등급 및 영향반경
    print("🛣️ 도로 정보 및 영향반경 계산 중...")
    df_enhanced['도로등급'] = df_enhanced['도로'].apply(classifier.get_road_grade)
    df_enhanced['영향반경_미터'] = df_enhanced['공사유형'].apply(classifier.get_impact_radius)
    
    road_grade_counts = df_enhanced['도로등급'].value_counts()
    print("✅ 도로 등급 분류 완료:")
    for grade, count in road_grade_counts.items():
        print(f"   - {grade}: {count:,}건")
    
    # 8. 최종 컬럼 정리
    final_cols = [
        # 기본 정보 (7개)
        '관리코드', '구', '동', '공사명', '공사명_정제', '처리상태',
        
        # 날짜 및 상태 (3개)
        '착공일', '준공일', '공사상태',
        
        # 위치 정보 (1개)
        '지오코딩주소',
        
        # 도로/포장 정보 (2개)
        '도로', '포장',
        
        # 공사 유형 및 분류 (2개)
        '공사유형', '공사유형_대분류',
        
        # 기본 위험도 (2개)
        '기본_위험도_레벨', '기본_위험도_카테고리',
        
        # 현재 위험도 (2개) 
        '현재_위험도_레벨', '현재_위험도_카테고리',
        
        # 도로 및 영향반경 (2개)
        '도로등급', '영향반경_미터',
        
        # 음성 안내 관련 (3개)
        '안내_우선순위', '안내_활성화_여부', '안내_메시지'
    ]
    
    df_final = df_enhanced[final_cols].copy()
    
    print(f"\n📊 개선된 데이터 생성 완료:")
    print(f"   - 최종 데이터: {len(df_final):,}건")
    print(f"   - 최종 컬럼: {len(final_cols)}개 (기존 19개 → {len(final_cols)}개)")
    print(f"   - 새로 추가된 컬럼: 상태별 차등 위험도, 안내 우선순위, 안내 메시지")
    
    return df_final

# 개선된 전처리 실행
if df_raw is not None:
    df_enhanced = create_enhanced_construction_data(df_raw)
    
    if df_enhanced is not None:
        print(f"\n📋 개선된 데이터 미리보기:")
        display_cols = ['관리코드', '구', '공사유형', '공사상태', '기본_위험도_레벨', '현재_위험도_레벨', '현재_위험도_카테고리', '안내_활성화_여부']
        print(df_enhanced[display_cols].head(5))

# =============================================================================
# 4. 지오코딩 (기존 코드와 동일)
# =============================================================================

class SimpleKakaoGeocoder:
    """카카오맵 지오코딩 서비스 (병렬처리 최적화)"""
    
    def __init__(self):
        self.api_key = os.getenv('KAKAO_REST_API_KEY')
        self.base_url = "https://dapi.kakao.com/v2/local/search/address.json"
        self.delay = 0.1  # 병렬처리로 인한 딜레이 감소
        self.session = requests.Session()  # 세션 재사용으로 성능 향상
        self.lock = Lock()  # 스레드 안전성
        self.request_count = 0
        self.success_count = 0
        
        # 서울시 자치구별 중심 좌표
        self.district_coords = {
            '강남구': (37.4979, 127.0276), '강동구': (37.5301, 127.1238),
            '강북구': (37.6398, 127.0256), '강서구': (37.5509, 126.8495),
            '관악구': (37.4784, 126.9516), '광진구': (37.5385, 127.0823),
            '구로구': (37.4955, 126.8874), '금천구': (37.4563, 126.8956),
            '노원구': (37.6542, 127.0568), '도봉구': (37.6688, 127.0471),
            '동대문구': (37.5744, 127.0396), '동작구': (37.5124, 126.9393),
            '마포구': (37.5637, 126.9084), '서대문구': (37.5794, 126.9368),
            '서초구': (37.4837, 127.0324), '성동구': (37.5634, 127.0365),
            '성북구': (37.5894, 127.0167), '송파구': (37.5145, 127.1059),
            '양천구': (37.5168, 126.8664), '영등포구': (37.5264, 126.8962),
            '용산구': (37.5326, 126.9905), '은평구': (37.6028, 126.9292),
            '종로구': (37.5735, 126.9788), '중구': (37.5641, 126.9979),
            '중랑구': (37.6063, 127.0925)
        }
    
    def get_coordinates_single(self, address, idx=None):
        """단일 주소를 좌표로 변환 (스레드 안전)"""
        with self.lock:
            self.request_count += 1
            
        if not self.api_key:
            return self._get_fallback_coordinates(address, idx)
            
        try:
            headers = {"Authorization": f"KakaoAK {self.api_key}"}
            params = {"query": address}
            
            # 병렬처리 시 너무 빠른 요청 방지
            time.sleep(self.delay)
            
            response = self.session.get(self.base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data['documents']:
                    result = data['documents'][0]
                    lat, lng = float(result['y']), float(result['x'])
                    
                    with self.lock:
                        self.success_count += 1
                    
                    if 37.4 <= lat <= 37.7 and 126.8 <= lng <= 127.2:
                        return {
                            'index': idx,
                            'lat': lat,
                            'lng': lng,
                            'status': 'API성공',
                            'address': address
                        }
                    else:
                        return {
                            'index': idx,
                            'lat': lat,
                            'lng': lng,
                            'status': 'API성공(범위외)',
                            'address': address
                        }
                else:
                    return self._get_fallback_coordinates(address, idx, 'API결과없음')
            elif response.status_code == 429:
                # 요청 한도 초과 시 잠시 대기 후 재시도
                time.sleep(2)
                return self.get_coordinates_single(address, idx)
            else:
                return self._get_fallback_coordinates(address, idx, f'API오류({response.status_code})')
                
        except Exception as e:
            return self._get_fallback_coordinates(address, idx, f'예외({str(e)[:20]})')
    
    def _get_fallback_coordinates(self, address, idx, status='API실패'):
        """백업 좌표 반환 (구별 중심점)"""
        for district, coords in self.district_coords.items():
            if district in address:
                lat, lng = coords
                # 약간의 랜덤 변화로 겹치지 않게
                lat += np.random.uniform(-0.01, 0.01)
                lng += np.random.uniform(-0.01, 0.01)
                return {
                    'index': idx,
                    'lat': lat,
                    'lng': lng,
                    'status': f'구중심({district})',
                    'address': address
                }
        
        # 구를 찾을 수 없으면 서울 중심
        return {
            'index': idx,
            'lat': 37.5665,
            'lng': 126.9780,
            'status': f'서울중심({status})',
            'address': address
        }
    
    def get_statistics(self):
        """API 호출 통계 반환"""
        return {
            'total_requests': self.request_count,
            'successful_requests': self.success_count,
            'success_rate': (self.success_count / self.request_count * 100) if self.request_count > 0 else 0
        }

def convert_addresses_to_coordinates_parallel(df, max_workers=10, batch_size=100, sample_size=None):
    """병렬 지오코딩 실행"""
    df_coord = df.copy()
    geocoder = SimpleKakaoGeocoder()
    
    if sample_size and sample_size < len(df_coord):
        print(f"🧪 테스트용 샘플링: {sample_size}건만 처리")
        df_coord = df_coord.head(sample_size)
    
    print(f"\n🗺️ 병렬 지오코딩 시작... (총 {len(df_coord):,}건)")
    print(f"⚡ 병렬 작업자: {max_workers}개 스레드")
    print(f"📦 배치 크기: {batch_size}건씩 처리")
    
    if not geocoder.api_key:
        print("⚠️ API 키 없음 → 구별 중심 좌표만 사용")
        max_workers = 1  # API 없을 때는 병렬처리 불필요
    
    # 주소와 인덱스 준비
    address_data = []
    for idx, row in df_coord.iterrows():
        address_data.append((row['지오코딩주소'], idx))
    
    # 배치별 병렬 처리
    all_results = []
    total_batches = (len(address_data) + batch_size - 1) // batch_size
    
    start_time = time.time()
    
    for batch_num in range(total_batches):
        batch_start = batch_num * batch_size
        batch_end = min(batch_start + batch_size, len(address_data))
        batch_data = address_data[batch_start:batch_end]
        
        print(f"   📦 배치 {batch_num + 1}/{total_batches}: {batch_start + 1}-{batch_end} ({len(batch_data)}개)")
        
        # 병렬 처리
        batch_results = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 작업 제출
            future_to_data = {
                executor.submit(geocoder.get_coordinates_single, address, idx): (address, idx)
                for address, idx in batch_data
            }
            
            # 결과 수집
            for future in as_completed(future_to_data):
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as e:
                    address, idx = future_to_data[future]
                    print(f"   ❌ 스레드 오류 {idx}: {str(e)[:50]}")
                    # 오류 시 백업 좌표 사용
                    backup = geocoder._get_fallback_coordinates(address, idx, '스레드오류')
                    batch_results.append(backup)
        
        all_results.extend(batch_results)
        
        # 배치 간 잠시 대기 (API 안정성)
        if batch_num < total_batches - 1 and geocoder.api_key:
            time.sleep(0.5)
        
        # 진행률 표시
        elapsed_time = time.time() - start_time
        processed = len(all_results)
        if processed > 0:
            estimated_total_time = elapsed_time * len(address_data) / processed
            remaining_time = estimated_total_time - elapsed_time
            print(f"   ⏱️ 진행률: {processed}/{len(address_data)} ({processed/len(address_data)*100:.1f}%) | "
                  f"경과: {elapsed_time:.1f}초 | 예상 남은시간: {remaining_time:.1f}초")
    
    # 결과 정리 및 병합
    print("\n🔄 결과 병합 중...")
    results_df = pd.DataFrame(all_results)
    
    # 인덱스 기준으로 원본 데이터와 병합
    df_result = df_coord.merge(results_df, left_index=True, right_on='index', how='left')
    df_result.drop(['index', 'address'], axis=1, inplace=True)
    df_result.rename(columns={'lat': '위도', 'lng': '경도', 'status': '좌표상태'}, inplace=True)
    
    # 통계 출력
    total_time = time.time() - start_time
    stats = geocoder.get_statistics()
    
    print("✅ 병렬 지오코딩 완료:")
    print(f"   ⏱️ 총 처리시간: {total_time:.1f}초")
    print(f"   🚀 평균 처리속도: {len(df_coord)/total_time:.1f}건/초")
    print(f"   📊 API 통계: {stats['total_requests']}회 요청, {stats['success_rate']:.1f}% 성공")
    
    # 좌표 상태별 분포
    status_counts = df_result['좌표상태'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_result) * 100) if len(df_result) > 0 else 0
        print(f"   - {status}: {count:,}건 ({percentage:.1f}%)")
    
    return df_result

# 지오코딩 실행 (병렬처리)
if 'df_enhanced' in locals() and df_enhanced is not None:
    print("\n💡 전체 데이터를 병렬처리로 처리합니다.")
    print("   ⚡ 병렬처리로 속도가 크게 향상됩니다!")
    
    # 병렬처리 설정
    max_workers = 8  # 동시 스레드 수 (너무 높으면 API 제한에 걸릴 수 있음)
    batch_size = 50  # 배치당 처리 건수
    
    df_with_coords = convert_addresses_to_coordinates_parallel(
        df_enhanced, 
        max_workers=max_workers, 
        batch_size=batch_size, 
        sample_size=None
    )
    
    if df_with_coords is not None:
        print(f"\n📍 좌표 변환 결과 미리보기:")
        display_cols = ['관리코드', '구', '공사유형', '공사상태', '현재_위험도_레벨', '안내_활성화_여부', '위도', '경도', '좌표상태']
        print(df_with_coords[display_cols].head(5))
    else:
        print("❌ 지오코딩 실패!")
else:
    print("❌ df_enhanced 데이터가 없습니다!")

# =============================================================================
# 5. 프로젝트별 집계 테이블 생성
# =============================================================================

def create_project_summary_table(df_with_coords):
    """프로젝트별 집계 테이블 생성 (중복 제거)"""
    print("\n📊 프로젝트별 집계 테이블 생성 중...")
    
    # 관리코드별 집계
    project_summary = []
    
    for code, group in df_with_coords.groupby('관리코드'):
        # 기본 정보 (첫 번째 행 기준)
        first_row = group.iloc[0]
        
        # 좌표 집계
        valid_coords = group.dropna(subset=['위도', '경도'])
        
        if len(valid_coords) > 0:
            center_lat = valid_coords['위도'].mean()
            center_lng = valid_coords['경도'].mean()
            
            # 최대 영향반경 계산 (중심점에서 가장 먼 구간까지의 거리)
            if len(valid_coords) > 1:
                distances = []
                for _, row in valid_coords.iterrows():
                    # 간단한 유클리드 거리 (실제로는 haversine 공식 사용 권장)
                    dist = np.sqrt((row['위도'] - center_lat)**2 + (row['경도'] - center_lng)**2) * 111000  # 대략적 미터 변환
                    distances.append(dist)
                max_impact_radius = max(distances) if distances else first_row['영향반경_미터']
            else:
                max_impact_radius = first_row['영향반경_미터']
        else:
            center_lat = center_lng = None
            max_impact_radius = first_row['영향반경_미터']
        
        # 가장 신뢰도 높은 좌표상태 선택
        coord_status_priority = {'API성공': 1, 'API성공(범위외)': 2, '구중심': 3, '서울중심': 4}
        best_status = min(group['좌표상태'].dropna(), 
                         key=lambda x: coord_status_priority.get(x.split('(')[0], 5))
        
        project_info = {
            # 기본 정보
            '관리코드': code,
            '구': first_row['구'],
            '동': first_row['동'],
            '공사명': first_row['공사명'],
            '공사명_정제': first_row['공사명_정제'],
            '처리상태': first_row['처리상태'],
            
            # 날짜 및 상태
            '착공일': first_row['착공일'],
            '준공일': first_row['준공일'],
            '공사상태': first_row['공사상태'],
            
            # 집계된 위치 정보
            '중심위도': center_lat,
            '중심경도': center_lng,
            '구간수': len(group),
            '최대영향반경': max_impact_radius,
            
            # 도로/포장 정보
            '도로': first_row['도로'],
            '포장': first_row['포장'],
            
            # 분류 및 위험도 정보
            '공사유형': first_row['공사유형'],
            '공사유형_대분류': first_row['공사유형_대분류'],
            '기본_위험도_레벨': first_row['기본_위험도_레벨'],
            '기본_위험도_카테고리': first_row['기본_위험도_카테고리'],
            '현재_위험도_레벨': first_row['현재_위험도_레벨'],
            '현재_위험도_카테고리': first_row['현재_위험도_카테고리'],
            '도로등급': first_row['도로등급'],
            '영향반경_미터': first_row['영향반경_미터'],
            
            # 음성 안내 정보
            '안내_우선순위': first_row['안내_우선순위'],
            '안내_활성화_여부': first_row['안내_활성화_여부'],
            '안내_메시지': first_row['안내_메시지'],
            
            # 좌표 상태
            '최신_좌표상태': best_status
        }
        
        project_summary.append(project_info)
    
    df_projects = pd.DataFrame(project_summary)
    
    print(f"✅ 프로젝트 집계 완료:")
    print(f"   - 원본 행수: {len(df_with_coords):,}건")
    print(f"   - 집계 후: {len(df_projects):,}건 (중복 제거)")
    print(f"   - 중복률: {(1 - len(df_projects)/len(df_with_coords))*100:.1f}%")
    
    return df_projects

def create_segment_detail_table(df_with_coords):
    """구간 상세 테이블 생성"""
    print("\n📍 구간 상세 테이블 생성 중...")
    
    # 구간 ID 생성
    df_segments = df_with_coords.copy()
    df_segments['구간_ID'] = range(1, len(df_segments) + 1)
    
    # 구간별 순서 생성 (관리코드 내에서)
    df_segments['구간_순서'] = df_segments.groupby('관리코드').cumcount() + 1
    
    # 필요한 컬럼만 선택
    segment_cols = [
        '관리코드', '구간_ID', '구간_순서',
        '지오코딩주소', '위도', '경도', '좌표상태'
    ]
    
    df_segments_final = df_segments[segment_cols].copy()
    
    print(f"✅ 구간 상세 테이블 생성 완료:")
    print(f"   - 총 구간수: {len(df_segments_final):,}개")
    print(f"   - 컬럼수: {len(segment_cols)}개")
    
    return df_segments_final

# 집계 테이블 생성
if 'df_with_coords' in locals() and df_with_coords is not None:
    print("\n🔍 집계 테이블 생성 시작...")
    df_projects = create_project_summary_table(df_with_coords)
    df_segments = create_segment_detail_table(df_with_coords)
    
    if df_projects is not None and df_segments is not None:
        print(f"\n📋 프로젝트 요약 테이블 미리보기:")
        project_display_cols = ['관리코드', '구', '공사유형', '공사상태', '현재_위험도_레벨', '안내_활성화_여부', '구간수', '중심위도', '중심경도']
        print(df_projects[project_display_cols].head(3))
        
        print(f"\n📋 구간 상세 테이블 미리보기:")
        segment_display_cols = ['관리코드', '구간_ID', '구간_순서', '위도', '경도']
        print(df_segments[segment_display_cols].head(3))
    else:
        print("❌ 집계 테이블 생성 실패!")
else:
    print("❌ df_with_coords 데이터가 없습니다!")

# =============================================================================
# 6. 최종 검증 및 저장
# =============================================================================

def validate_and_save_enhanced_data(df_projects, df_segments, df_combined, output_dir="../../data/processed"):
    """개선된 데이터 검증 및 저장"""
    print(f"\n🔍 최종 데이터 검증...")
    
    # 1. 기본 통계
    total_projects = len(df_projects) if df_projects is not None else 0
    total_segments = len(df_segments) if df_segments is not None else 0
    total_combined = len(df_combined) if df_combined is not None else 0
    
    print("=" * 60)
    print("📊 최종 데이터 요약")
    print("=" * 60)
    print(f"🎯 프로젝트 요약: {total_projects:,}건")
    print(f"📍 구간 상세: {total_segments:,}건")
    print(f"🗂️ 통합 테이블: {total_combined:,}건")
    
    if df_projects is not None:
        # 현재 위험도 분포 (상태별)
        print(f"\n🚨 현재 위험도별 분포:")
        risk_dist = df_projects['현재_위험도_카테고리'].value_counts()
        for category, count in risk_dist.items():
            percentage = (count / total_projects * 100) if total_projects > 0 else 0
            print(f"   - {category}: {count:,}건 ({percentage:.1f}%)")
        
        # 공사 유형 분포
        print(f"\n🏗️ 공사 유형별 분포:")
        type_dist = df_projects['공사유형'].value_counts()
        for type_name, count in type_dist.head(5).items():
            percentage = (count / total_projects * 100) if total_projects > 0 else 0
            print(f"   - {type_name}: {count:,}건 ({percentage:.1f}%)")
        
        # 음성 안내 활성화 상태
        active_guidance = (df_projects['안내_활성화_여부'] == True).sum() if '안내_활성화_여부' in df_projects.columns else 0
        print(f"\n📢 음성 안내 활성화: {active_guidance:,}건 / {total_projects:,}건 ({active_guidance/total_projects*100:.1f}%)")
        
        # 상태별 진행중인 공사
        ongoing_projects = (df_projects['공사상태'] == '진행중').sum() if '공사상태' in df_projects.columns else 0
        planned_projects = (df_projects['공사상태'] == '미착공').sum() if '공사상태' in df_projects.columns else 0
        completed_projects = (df_projects['공사상태'] == '완료').sum() if '공사상태' in df_projects.columns else 0
        
        print(f"\n🚧 공사 상태별 분포:")
        print(f"   - 진행중: {ongoing_projects:,}건")
        print(f"   - 미착공: {planned_projects:,}건") 
        print(f"   - 완료: {completed_projects:,}건")
    
    # 2. 데이터 저장 (latest 버전만)
    try:
        os.makedirs(output_dir, exist_ok=True)
        
        saved_files = []
        
        # 프로젝트 요약 테이블 저장
        if df_projects is not None:
            projects_latest = os.path.join(output_dir, "road_excavation_projects_latest.csv")
            df_projects.to_csv(projects_latest, index=False, encoding='utf-8-sig')
            saved_files.append(projects_latest)
        
        # 구간 상세 테이블 저장
        if df_segments is not None:
            segments_latest = os.path.join(output_dir, "road_excavation_segments_latest.csv")
            df_segments.to_csv(segments_latest, index=False, encoding='utf-8-sig')
            saved_files.append(segments_latest)
        
        # 통합 테이블 저장
        if df_combined is not None:
            combined_latest = os.path.join(output_dir, "road_excavation_combined_latest.csv")
            df_combined.to_csv(combined_latest, index=False, encoding='utf-8-sig')
            saved_files.append(combined_latest)
        
        print(f"\n💾 데이터 저장 완료:")
        for file_path in saved_files:
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"   - {os.path.basename(file_path)} ({file_size_mb:.2f} MB)")
        
        return {
            'total_projects': total_projects,
            'total_segments': total_segments,
            'total_combined': total_combined,
            'saved_files': saved_files
        }
        
    except Exception as e:
        print(f"❌ 저장 실패: {str(e)}")
        return None

# 최종 검증 및 저장
print("\n🔍 최종 저장 단계 확인...")

# 변수 존재 여부를 더 확실하게 체크
def check_dataframe_exists(df_name, df_var):
    """데이터프레임 존재 여부와 상태 확인"""
    try:
        if df_var is not None and len(df_var) > 0:
            print(f"✅ {df_name}: {len(df_var)}건 존재")
            return True
        else:
            print(f"❌ {df_name}: 데이터가 없거나 빈 상태")
            return False
    except Exception as e:
        print(f"❌ {df_name}: 오류 - {str(e)}")
        return False

# 각 데이터프레임 상태 확인
try:
    projects_ok = check_dataframe_exists("df_projects", df_projects)
    segments_ok = check_dataframe_exists("df_segments", df_segments) 
    coords_ok = check_dataframe_exists("df_with_coords", df_with_coords)
    
    print(f"\n📊 데이터프레임 상태 요약:")
    print(f"   - 프로젝트 요약: {'✅' if projects_ok else '❌'}")
    print(f"   - 구간 상세: {'✅' if segments_ok else '❌'}")
    print(f"   - 통합 데이터: {'✅' if coords_ok else '❌'}")
    
    if projects_ok and segments_ok and coords_ok:
        print("\n🎉 모든 데이터프레임이 정상적으로 생성되었습니다!")
        print("💾 저장을 시작합니다...")
        
        try:
            final_summary = validate_and_save_enhanced_data(df_projects, df_segments, df_with_coords)
            
            if final_summary:
                print(f"\n🎉 FR-005 개선된 데이터 전처리 완료!")
                print("=" * 70)
                print("📋 최종 데이터셋 정보:")
                print(f"   - 프로젝트 요약: {final_summary['total_projects']:,}건")
                print(f"   - 구간 상세: {final_summary['total_segments']:,}건")
                print(f"   - 통합 데이터: {final_summary['total_combined']:,}건")
                print(f"   - 저장 파일: {len(final_summary['saved_files'])}개")
                
                print("\n📁 저장된 파일 목록:")
                for file_path in final_summary['saved_files']:
                    print(f"   - {file_path}")
                
                print("\n🚀 다음 단계:")
                print("   1. 음성 안내 메시지 템플릿 개발")
                print("   2. 위험도별 안전거리 설정")
                print("   3. 실시간 위치 기반 필터링 로직")
                print("   4. Azure Speech Services 연동")
                print("   5. 경로 안내 시스템 구축")
                
                print("\n📱 FR-005 음성 안내 활용 예시:")
                if len(df_projects) > 0:
                    # 현재 진행중인 고위험 공사 우선 표시
                    high_risk_ongoing = df_projects[
                        (df_projects['공사상태'] == '진행중') & 
                        (df_projects['현재_위험도_레벨'] >= 3.0)
                    ]
                    
                    if len(high_risk_ongoing) > 0:
                        sample = high_risk_ongoing.iloc[0]
                        print(f"   🚨 진행중 고위험 공사 예시:")
                        print(f"   📍 위치: {sample['구']} {sample['동']}")
                        print(f"   🏗️ 유형: {sample['공사유형']} (기본위험도: {sample['기본_위험도_레벨']}, 현재위험도: {sample['현재_위험도_레벨']})")
                        print(f"   📢 안내: {sample['안내_메시지']}")
                    else:
                        # 예정된 공사 예시
                        planned_projects = df_projects[df_projects['공사상태'] == '미착공']
                        if len(planned_projects) > 0:
                            sample = planned_projects.iloc[0]
                            print(f"   📅 예정 공사 예시:")
                            print(f"   📍 위치: {sample['구']} {sample['동']}")
                            print(f"   🏗️ 유형: {sample['공사유형']} (기본위험도: {sample['기본_위험도_레벨']}, 현재위험도: {sample['현재_위험도_레벨']})")
                            print(f"   📢 안내: {sample['안내_메시지']}")
                        else:
                            print("   ℹ️ 현재 활성화된 안내 대상이 없습니다.")
            else:
                print("❌ 저장 과정에서 오류가 발생했습니다.")
        except Exception as e:
            print(f"❌ 저장 중 예외 발생: {str(e)}")
            import traceback
            traceback.print_exc()
    else:
        print("\n❌ 일부 데이터프레임 생성에 실패했습니다.")
        print("💡 위의 단계들을 다시 확인해보세요.")

except NameError as e:
    print(f"❌ 변수 오류: {str(e)}")
    print("💡 이전 단계에서 일부 변수가 생성되지 않았습니다.")
except Exception as e:
    print(f"❌ 예상치 못한 오류: {str(e)}")
    import traceback
    traceback.print_exc()

# =============================================================================
# 7. 데이터 분석 및 인사이트
# =============================================================================

def analyze_construction_patterns(df_projects):
    """공사 패턴 심화 분석 (개선된 위험도 시스템 적용)"""
    
    if df_projects is None or len(df_projects) == 0:
        print("❌ 분석할 프로젝트 데이터가 없습니다.")
        return
    
    print(f"\n📊 공사 패턴 심화 분석 (상태별 차등 위험도 적용)...")
    print("=" * 60)
    
    # 1. 상태별 위험도 분포
    print("🗺️ 상태별 현재 위험도 분포:")
    status_risk_summary = df_projects.groupby(['공사상태', '현재_위험도_카테고리']).size().unstack(fill_value=0)
    for status in ['진행중', '미착공', '완료', '불명']:
        if status in status_risk_summary.index:
            row_data = status_risk_summary.loc[status]
            total = row_data.sum()
            print(f"   📋 {status}: {total}건")
            for category, count in row_data[row_data > 0].items():
                print(f"      • {category}: {count}건")
    
    # 2. 현재 진행중인 고위험 공사
    high_risk_ongoing = df_projects[
        (df_projects['공사상태'] == '진행중') & 
        (df_projects['현재_위험도_레벨'] >= 3.0)
    ]
    
    print(f"\n🚧 현재 진행중인 고위험 공사: {len(high_risk_ongoing)}건")
    if len(high_risk_ongoing) > 0:
        print("   🚨 지역별 분포:")
        top_ongoing = high_risk_ongoing['구'].value_counts().head(5)
        for district, count in top_ongoing.items():
            avg_risk = high_risk_ongoing[high_risk_ongoing['구'] == district]['현재_위험도_레벨'].mean()
            print(f"      - {district}: {count}건 (평균 위험도: {avg_risk:.1f})")
    
    # 3. 예정된 공사 중 주의 대상
    planned_high_risk = df_projects[
        (df_projects['공사상태'] == '미착공') & 
        (df_projects['기본_위험도_레벨'] >= 4)
    ]
    
    print(f"\n📅 예정된 고위험 공사: {len(planned_high_risk)}건")
    if len(planned_high_risk) > 0:
        print("   ⚠️ 주의 대상 지역:")
        top_planned = planned_high_risk['구'].value_counts().head(3)
        for district, count in top_planned.items():
            print(f"      - {district}: {count}건")
    
    # 4. 음성 안내 우선순위 분석
    print(f"\n📢 음성 안내 우선순위 분석:")
    priority_high = (df_projects['안내_우선순위'] >= 10).sum()
    priority_medium = ((df_projects['안내_우선순위'] >= 7) & (df_projects['안내_우선순위'] < 10)).sum()
    priority_low = (df_projects['안내_우선순위'] < 7).sum()
    
    print(f"   🔥 최우선 안내 (≥10): {priority_high}건")
    print(f"   🟡 중요 안내 (7-9): {priority_medium}건")
    print(f"   🔵 일반 안내 (<7): {priority_low}건")
    
    # 5. 개선된 안내 메시지 예시
    print(f"\n📱 상태별 안내 메시지 예시:")
    
    # 진행중 공사 예시
    ongoing_sample = df_projects[df_projects['공사상태'] == '진행중']
    if len(ongoing_sample) > 0:
        sample = ongoing_sample.iloc[0]
        print(f"   🚧 진행중: {sample['안내_메시지']}")
    
    # 미착공 공사 예시
    planned_sample = df_projects[df_projects['공사상태'] == '미착공']
    if len(planned_sample) > 0:
        sample = planned_sample.iloc[0]
        print(f"   📅 미착공: {sample['안내_메시지']}")
    
    # 완료 공사 예시
    completed_sample = df_projects[df_projects['공사상태'] == '완료']
    if len(completed_sample) > 0:
        sample = completed_sample.iloc[0]
        print(f"   ✅ 완료: {sample['안내_메시지']}")
    
    # 6. 위험도 변화 효과 분석
    print(f"\n📈 위험도 차등 적용 효과:")
    basic_high_risk = (df_projects['기본_위험도_레벨'] >= 4).sum()
    current_high_risk = (df_projects['현재_위험도_레벨'] >= 4.0).sum()
    
    print(f"   📊 기본 고위험(≥4): {basic_high_risk}건")
    print(f"   📊 현재 고위험(≥4.0): {current_high_risk}건")
    print(f"   📉 위험도 감소: {basic_high_risk - current_high_risk}건 ({(basic_high_risk - current_high_risk)/basic_high_risk*100:.1f}%)")
    
    return df_projects

# 패턴 분석 실행
if 'df_projects' in locals() and df_projects is not None:
    priority_data = analyze_construction_patterns(df_projects)

print("\n" + "=" * 70)
print("🏁 FR-005 개선된 전처리 스크립트 실행 완료")
print("=" * 70)
print("💡 생성된 파일들을 확인하고 음성 안내 로직 개발을 시작하세요!")
print("📧 문의사항이 있으시면 개발팀에 연락 부탁드립니다.")

🚧 FR-005: 도로굴착 공사현황 데이터 전처리 (개선 버전)
🎯 위험도 분류 및 분석 포함
🔐 환경변수 로드 완료
🔍 카카오 API 연결 테스트...
🔑 API 키: 7730cd19fd...
✅ API 연결 성공!
   📍 테스트 결과: 위도 37.5173319258532, 경도 127.047377408384
🌐 API 사용 가능: Yes
----------------------------------------------------------------------

📂 데이터 로드 시작...
✅ 파일 로드 성공 (UTF-8)
📊 총 데이터: 533건
📋 컬럼: 10개
📅 컬럼 목록: ['관리코드', '구', '동', '공사명', '착공일 ~ 준공일', '신청자', '처리상태', '도로', '포장', '허가번호']

📋 데이터 미리보기 (첫 3행):
           관리코드    구    동                                                공사명  \
0  일반-210728-75  서초구  잠원동  잠원동 신반포14차조합(롯데건설)3000kW 신설_보도(대선, 송영아)<BR>(서초...   
1  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   
2  일반-210527-84  용산구  동자동  동암간53L3외 노후경사전주 교체공사(361120213331,이엔티,박철)<BR>(...   

                 착공일 ~ 준공일  
0  2022-06-20 ~ 2022-08-31  
1  2022-06-16 ~ 2022-08-14  
2  2022-06-16 ~ 2022-08-14  

🏢 자치구별 분포 (상위 10개):
   - 서초구: 192건
   - 강남구: 94건
   - 성북구: 37건
   - 관악구: 33건
   - 강동구: 33건
   - 강북구: 19건
   - 종로구: 19건
   - 도봉구: 18건
   

In [17]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv('../../data/processed/road_excavation_combined_latest.csv')

# '공사명', '처리상태' 컬럼 제거
df = df.drop(columns=['공사명', '처리상태','공사유형_대분류', '공사명_정제'])

# 중복 행 제거
df = df.drop_duplicates()

# 위도, 경도 → 소수점 4자리 버림 처리
df['위도'] = np.floor(df['위도'] * 10000) / 10000
df['경도'] = np.floor(df['경도'] * 10000) / 10000
df = df.drop_duplicates()

# 두 가지 인코딩 방식으로 저장
df.to_csv('공사현황_정제_utf8.csv', index=False, encoding='utf-8-sig')

print("✅")



✅
